<a href="https://colab.research.google.com/github/jeremydouti2-hub/Data-Science-Projects/blob/main/Multi_Domain_Data_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

# Charger les datasets
df1 = pd.read_csv("modular 5_1.csv")
df2 = pd.read_csv("modular 5_2.csv")
df3 = pd.read_csv("modular 5_3.csv")

# Afficher les premières lignes
print("=== Dataset 1 ===")
display(df1.head())

print("=== Dataset 2 ===")
display(df2.head())

print("=== Dataset 3 ===")
display(df3.head())

In [ ]:
# General information
print("=== DATASET 1 INFO ===")
df1.info()

print("=== DATASET 2 INFO ===")
df2.info()

print("=== DATASET 3 INFO ===")
df3.info()

# Statistics
print("=== DATASET 1 STATS ===")
display(df1.describe())

print("=== DATASET 2 STATS ===")
display(df2.describe())

print("=== DATASET 3 STATS ===")
display(df3.describe())

In [ ]:
# ============================================================
# SECTION 1: ASSOCIATION, CORRELATION & REGRESSION ANALYSIS
# Dataset 1 – California Housing Prices
# Student: Gountante Douti | PRN: 31250500042
# ============================================================

# ── 0. IMPORTS & GLOBAL SETTINGS
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from scipy.stats import pearsonr, spearmanr, kendalltau

from sklearn.linear_model import (LinearRegression, Ridge, Lasso,
                                   ElasticNet, LogisticRegression)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                              r2_score, classification_report,
                              confusion_matrix)
from sklearn.pipeline import Pipeline

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 100, 'figure.figsize': (10, 6),
                     'axes.titlesize': 13, 'axes.labelsize': 11})
print(" All libraries loaded.")

# ── 1. LOAD DATA ────
df1 = pd.read_csv("modular 5_1.csv")   # California Housing
df2 = pd.read_csv("modular 5_2.csv")   # Superstore Sales
df3 = pd.read_csv("modular 5_3.csv")   # Mall Customers

print(f"\nDataset 1 shape : {df1.shape}")
print(f"Dataset 2 shape : {df2.shape}")
print(f"Dataset 3 shape : {df3.shape}")

# ── 2. PREPROCESSING – DATASET 1 ─
df1_enc = df1.copy()
df1_enc['ocean_proximity'] = LabelEncoder().fit_transform(
    df1_enc['ocean_proximity'].astype(str))

df1_clean = df1_enc.dropna().reset_index(drop=True)
print(f"\nDF1 after cleaning: {df1_clean.shape}  "
      f"(dropped {len(df1_enc) - len(df1_clean)} rows with NaN)")

FEATURES = ['longitude','latitude','housing_median_age','total_rooms',
            'total_bedrooms','population','households','median_income',
            'ocean_proximity']
TARGET = 'median_house_value'

X_raw = df1_clean[FEATURES]
y     = df1_clean[TARGET]

# ── 3. DESCRIPTIVE STATISTICS ─
print("\n===== DESCRIPTIVE STATISTICS – Dataset 1 =====")
print(df1_clean.describe().round(2).to_string())

# ── 4. CORRELATION ANALYSIS (Pearson / Spearman / Kendall) ───
print("\n===== CORRELATION ANALYSIS =====")

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

for ax, method, title in zip(axes,
                              ['pearson', 'spearman', 'kendall'],
                              ['Pearson r', 'Spearman ρ', 'Kendall τ']):
    c = df1_clean.corr(method=method)
    sns.heatmap(c, annot=True, fmt='.2f', cmap='RdYlGn',
                center=0, square=True, linewidths=0.3,
                annot_kws={'size': 7}, ax=ax)
    ax.set_title(f'{title} Correlation Matrix', fontsize=11)
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)

plt.suptitle('Dataset 1 – Correlation Matrices (Three Methods)',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('sec1_correlations.png', bbox_inches='tight')
plt.show()
print(" Correlation heatmaps saved.")

# Significance test for each feature vs target
print(f"\nCorrelation with '{TARGET}'  [Pearson | p-value]:")
for col in FEATURES:
    r, p = pearsonr(df1_clean[col], y)
    stars = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    print(f"  {col:<25}  r={r:+.4f}   p={p:.2e}  {stars}")

# ── 5. SIMPLE LINEAR REGRESSION (median_income → house_value) ─
print("\n===== SIMPLE LINEAR REGRESSION =====")
x_simple = df1_clean[['median_income']]
X_tr, X_te, y_tr, y_te = train_test_split(x_simple, y,
                                            test_size=0.2, random_state=42)
slr = LinearRegression().fit(X_tr, y_tr)
y_pred_slr = slr.predict(X_te)

#  FIX: np.sqrt(mse) instead of squared=False
rmse_slr = np.sqrt(mean_squared_error(y_te, y_pred_slr))

print(f"  Intercept  : {slr.intercept_:,.2f}")
print(f"  Coefficient: {slr.coef_[0]:,.2f}")
print(f"  R²  (test) : {r2_score(y_te, y_pred_slr):.4f}")
print(f"  RMSE(test) : {rmse_slr:,.2f}")
print(f"  MAE (test) : {mean_absolute_error(y_te, y_pred_slr):,.2f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X_te, y_te, alpha=0.25, s=10, label='Actual')
x_line = np.linspace(X_te.values.min(), X_te.values.max(), 200).reshape(-1, 1)
ax.plot(x_line, slr.predict(x_line), 'r-', lw=2, label='Regression line')
ax.set_xlabel('Median Income')
ax.set_ylabel('Median House Value')
ax.set_title('Simple Linear Regression: Income → House Value')
ax.legend()
plt.tight_layout()
plt.savefig('sec1_slr.png', bbox_inches='tight')
plt.show()

# ── 6. MULTIPLE LINEAR REGRESSION (statsmodels OLS + diagnostics) ─
print("\n===== MULTIPLE LINEAR REGRESSION (statsmodels OLS) =====")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
X_scaled_df = pd.DataFrame(X_scaled, columns=FEATURES)
X_sm = sm.add_constant(X_scaled_df)

ols_model = sm.OLS(y.values, X_sm).fit()
print(ols_model.summary())

# VIF – multicollinearity
print("\n  Variance Inflation Factors:")
vif_data = pd.DataFrame({
    'Feature': FEATURES,
    'VIF': [variance_inflation_factor(X_scaled, i)
            for i in range(X_scaled.shape[1])]
})
print(vif_data.sort_values('VIF', ascending=False).to_string(index=False))

# Breusch-Pagan heteroscedasticity test
bp_stat, bp_p, _, _ = het_breuschpagan(ols_model.resid, X_sm)
print(f"\n  Breusch-Pagan test: stat={bp_stat:.4f}  p={bp_p:.4f}",
      "(heteroscedasticity present)" if bp_p < 0.05 else "(homoscedastic – OK)")

# Residual diagnostic plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fitted = ols_model.fittedvalues
resid  = ols_model.resid

axes[0].scatter(fitted, resid, alpha=0.2, s=8)
axes[0].axhline(0, color='red', lw=1.5)
axes[0].set(xlabel='Fitted values', ylabel='Residuals',
            title='Residuals vs Fitted')

sm.qqplot(resid, line='s', ax=axes[1], alpha=0.3)
axes[1].set_title('Q-Q Plot of Residuals')

axes[2].hist(resid, bins=50, edgecolor='black', color='steelblue')
axes[2].set(xlabel='Residual', ylabel='Count', title='Residual Distribution')

plt.suptitle('OLS Residual Diagnostics', fontsize=13)
plt.tight_layout()
plt.savefig('sec1_ols_diagnostics.png', bbox_inches='tight')
plt.show()
print(" OLS diagnostics saved.")

# ── 7. SKLEARN MULTIPLE REGRESSION (train/test + cross-validation) ─
print("\n===== SKLEARN MULTIPLE REGRESSION =====")
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_scaled, y,
                                                test_size=0.2, random_state=42)
mlr = LinearRegression().fit(X_tr2, y_tr2)
y_pred_mlr = mlr.predict(X_te2)

#  FIX: np.sqrt(mse) instead of squared=False
rmse_mlr = np.sqrt(mean_squared_error(y_te2, y_pred_mlr))

cv_scores = cross_val_score(LinearRegression(), X_scaled, y,
                             cv=KFold(n_splits=5, shuffle=True, random_state=42),
                             scoring='r2')
print(f"  Test  R²   : {r2_score(y_te2, y_pred_mlr):.4f}")
print(f"  Test  RMSE : {rmse_mlr:,.2f}")
print(f"  5-Fold CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Coefficient importance bar chart
coef_df = pd.Series(mlr.coef_, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
coef_df.plot.barh(ax=ax,
                  color=['tomato' if v < 0 else 'steelblue' for v in coef_df])
ax.axvline(0, color='black', lw=0.8)
ax.set(title='Multiple Regression – Standardised Coefficients',
       xlabel='Coefficient value')
plt.tight_layout()
plt.savefig('sec1_coef.png', bbox_inches='tight')
plt.show()

# ── 8. REGULARISED REGRESSION (Ridge / Lasso / ElasticNet)
print("\n===== REGULARISED REGRESSION =====")

ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0]
reg_results = []

for name, Model in [('Ridge', Ridge), ('Lasso', Lasso), ('ElasticNet', ElasticNet)]:
    best_r2, best_alpha, best_model = -np.inf, None, None
    for a in ALPHAS:
        kw = {'alpha': a, 'random_state': 42} if name == 'ElasticNet' \
             else {'alpha': a}
        m  = Model(**kw).fit(X_tr2, y_tr2)
        r2 = r2_score(y_te2, m.predict(X_te2))
        if r2 > best_r2:
            best_r2, best_alpha, best_model = r2, a, m

    #  FIX: np.sqrt(mse) instead of squared=False
    rmse_reg = np.sqrt(mean_squared_error(y_te2, best_model.predict(X_te2)))
    reg_results.append({'Model': name, 'Best Alpha': best_alpha,
                        'R² (test)': round(best_r2, 4),
                        'RMSE (test)': round(rmse_reg, 2)})
    print(f"  {name:<12} α={best_alpha:<6}  R²={best_r2:.4f}  RMSE={rmse_reg:,.2f}")

print("\n  Summary table:")
print(pd.DataFrame(reg_results).to_string(index=False))

# ── 9. LOGISTIC REGRESSION (binary classification)
print("\n===== LOGISTIC REGRESSION =====")
PRICE_THRESHOLD = y.quantile(0.75)
y_bin = (y >= PRICE_THRESHOLD).astype(int)
print(f"  Threshold (75th pct): ${PRICE_THRESHOLD:,.0f}  |  "
      f"Class balance: {y_bin.mean():.2%} high-value")

X_tr3, X_te3, y_tr3, y_te3 = train_test_split(X_scaled, y_bin,
                                                test_size=0.2,
                                                random_state=42,
                                                stratify=y_bin)
log_reg = LogisticRegression(C=0.1, max_iter=500,
                              solver='lbfgs', random_state=42)
log_reg.fit(X_tr3, y_tr3)
y_pred_log = log_reg.predict(X_te3)

print(f"\n  Classification Report:\n")
print(classification_report(y_te3, y_pred_log,
                             target_names=['Normal', 'High-Value']))

# Odds ratios
odds = np.exp(log_reg.coef_[0])
or_df = pd.Series(odds, index=FEATURES).sort_values(ascending=False)
print("\n  Odds Ratios (top 5):")
print(or_df.head(5).round(4).to_string())

# Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_te3, y_pred_log)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'High-Value'],
            yticklabels=['Normal', 'High-Value'], ax=ax)
ax.set(xlabel='Predicted', ylabel='Actual',
       title='Logistic Regression – Confusion Matrix')
plt.tight_layout()
plt.savefig('sec1_logit_cm.png', bbox_inches='tight')
plt.show()

# ── 10. SECTION SUMMARY ─
print("\n" + "="*60)
print("  SECTION 1 COMPLETE – Key Findings")
print("="*60)
print(f"  1. Strongest positive predictor : median_income")
print(f"     (Pearson r ≈ {pearsonr(df1_clean['median_income'], y)[0]:.3f})")
print(f"  2. OLS R²         : {ols_model.rsquared:.4f}")
print(f"  3. MLR CV R²      : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
best_reg = max(reg_results, key=lambda x: x['R² (test)'])
print(f"  4. Best reg. model: {best_reg['Model']}  (R²={best_reg['R² (test)']})")
print(f"  5. Logistic accuracy: {(y_pred_log == y_te3).mean():.2%}")
print("="*60)

In [ ]:
# ============================================================
# SECTION 2: PREDICTIVE ANALYTICS IMPLEMENTATION
# Dataset 2 – Superstore Sales (Sales Forecasting)
# Student: Gountante Douti | PRN: 31250500042
# ============================================================

# ── 0. IMPORTS & GLOBAL SETTINGS ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                               VotingRegressor, BaggingRegressor)
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import (train_test_split, cross_val_score,
                                      KFold, TimeSeriesSplit)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 100, 'figure.figsize': (12, 5),
                     'axes.titlesize': 13, 'axes.labelsize': 11})
print(" All Section 2 libraries loaded.")

# ============================================================
# 1. LOAD & INSPECT DATASET 2
# ============================================================
df2 = pd.read_csv("modular 5_2.csv")
print(f"\nDataset 2 shape : {df2.shape}")
print(f"Columns         : {list(df2.columns)}")
print(df2.head(3).to_string())

# ============================================================
# 2. PREPROCESSING
# ============================================================
df = df2.copy()

# --- 2-a  Parse dates -
# Try common date column names
for col in ['Order Date', 'order_date', 'OrderDate', 'date', 'Date']:
    if col in df.columns:
        df['Order Date'] = pd.to_datetime(df[col], dayfirst=True, errors='coerce')
        break

# --- 2-b  Identify sales column ------
sales_candidates = [c for c in df.columns
                    if any(k in c.lower() for k in ['sale','revenue','amount','profit'])]
SALES_COL = sales_candidates[0] if sales_candidates else df.select_dtypes(
    include=np.number).columns[0]
print(f"\n  → Sales column detected: '{SALES_COL}'")

# --- 2-c  Drop rows with missing sales / dates --
df = df.dropna(subset=[SALES_COL]).reset_index(drop=True)

# --- 2-d  Encode all remaining object columns ----------------
le = LabelEncoder()
cat_cols = df.select_dtypes(include='object').columns.tolist()
for c in cat_cols:
    df[c + '_enc'] = le.fit_transform(df[c].astype(str))

print(f"  Categorical cols encoded: {cat_cols}")
print(f"  Final shape after cleaning: {df.shape}")

# ============================================================
# 3. TIME-SERIES AGGREGATION (Monthly Sales)
# ============================================================
print("\n===== TIME-SERIES AGGREGATION =====")

if 'Order Date' in df.columns and df['Order Date'].notna().sum() > 10:
    ts = (df.set_index('Order Date')[SALES_COL]
            .resample('MS')          # Month-Start frequency
            .sum()
            .rename('Monthly_Sales'))
    ts = ts[ts > 0]
    print(f"  Monthly time series: {len(ts)} periods  "
          f"({ts.index.min().date()} → {ts.index.max().date()})")

    # Plot raw time series
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(ts.index, ts.values, marker='o', markersize=4, lw=1.8,
            color='steelblue', label='Monthly Sales')
    ax.fill_between(ts.index, ts.values, alpha=0.15, color='steelblue')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.xticks(rotation=45)
    ax.set(title='Monthly Sales – Time Series', ylabel='Sales ($)', xlabel='')
    ax.legend(); plt.tight_layout()
    plt.savefig('sec2_timeseries.png', bbox_inches='tight'); plt.show()
    TS_AVAILABLE = True
else:
    print("    No valid date column found – skipping time-series steps.")
    TS_AVAILABLE = False

# ============================================================
# 4. STATIONARITY TEST (ADF)
# ============================================================
if TS_AVAILABLE:
    print("\n===== AUGMENTED DICKEY-FULLER TEST =====")
    adf_result = adfuller(ts.values, autolag='AIC')
    print(f"  ADF Statistic : {adf_result[0]:.4f}")
    print(f"  p-value       : {adf_result[1]:.4f}")
    for k, v in adf_result[4].items():
        print(f"  Critical ({k})  : {v:.4f}")
    is_stationary = adf_result[1] < 0.05
    print(f"  → Series is {'STATIONARY ' if is_stationary else 'NON-STATIONARY   (differencing applied in SARIMA)'}")

    # ACF / PACF plots
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    plot_acf(ts.values,  lags=min(24, len(ts)//2 - 1), ax=axes[0], title='ACF')
    plot_pacf(ts.values, lags=min(24, len(ts)//2 - 1), ax=axes[1], method='ywm',
              title='PACF')
    plt.suptitle('Autocorrelation Analysis', fontsize=13)
    plt.tight_layout()
    plt.savefig('sec2_acf_pacf.png', bbox_inches='tight'); plt.show()

# ================================================
# 5. SARIMA FORECASTING
# ================================================
if TS_AVAILABLE and len(ts) >= 18:
    print("\n===== SARIMA FORECASTING =====")

    # Fixed parameters – safe and fast, no heavy grid search
    SARIMA_ORDER        = (1, 1, 1)      # p, d, q
    SARIMA_SEASONAL     = (1, 1, 0, 12)  # P, D, Q, s  (annual seasonality)
    FORECAST_HORIZON    = 6              # months ahead

    # Train / test split (last 6 months = test)
    split_idx   = len(ts) - FORECAST_HORIZON
    ts_train    = ts.iloc[:split_idx]
    ts_test     = ts.iloc[split_idx:]

    sarima_model = SARIMAX(ts_train,
                           order=SARIMA_ORDER,
                           seasonal_order=SARIMA_SEASONAL,
                           enforce_stationarity=False,
                           enforce_invertibility=False)
    sarima_fit = sarima_model.fit(disp=False, maxiter=200)
    print(sarima_fit.summary().tables[0])

    # Forecast
    forecast_obj = sarima_fit.get_forecast(steps=FORECAST_HORIZON)
    forecast_mean = forecast_obj.predicted_mean
    conf_int      = forecast_obj.conf_int(alpha=0.10)   # 90% CI

    # Metrics on test set
    rmse_sarima = np.sqrt(mean_squared_error(ts_test.values, forecast_mean.values))
    mae_sarima  = mean_absolute_error(ts_test.values, forecast_mean.values)
    mape_sarima = np.mean(np.abs((ts_test.values - forecast_mean.values) /
                                  (ts_test.values + 1e-9))) * 100
    print(f"\n  SARIMA Test RMSE : {rmse_sarima:,.2f}")
    print(f"  SARIMA Test MAE  : {mae_sarima:,.2f}")
    print(f"  SARIMA Test MAPE : {mape_sarima:.2f}%")

    # Plot forecast
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(ts_train.index, ts_train.values, label='Train', color='steelblue', lw=1.8)
    ax.plot(ts_test.index,  ts_test.values,  label='Actual Test',
            color='black', lw=2, linestyle='--')
    ax.plot(forecast_mean.index, forecast_mean.values,
            label='SARIMA Forecast', color='tomato', lw=2)
    ax.fill_between(conf_int.index,
                    conf_int.iloc[:, 0], conf_int.iloc[:, 1],
                    color='tomato', alpha=0.15, label='90% CI')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.xticks(rotation=45)
    ax.set(title=f'SARIMA{SARIMA_ORDER}x{SARIMA_SEASONAL} – 6-Month Forecast',
           ylabel='Sales ($)')
    ax.legend(); plt.tight_layout()
    plt.savefig('sec2_sarima_forecast.png', bbox_inches='tight'); plt.show()
    print(" SARIMA forecast saved.")
else:
    print("    Skipping SARIMA – not enough time periods (need ≥ 18).")

# ============================================================
# 6. FEATURE ENGINEERING FOR ML MODELS
# ============================================================
print("\n===== FEATURE ENGINEERING =====")

df_ml = df.copy()

# Date-based features (if date available)
if 'Order Date' in df_ml.columns and df_ml['Order Date'].notna().sum() > 10:
    df_ml['year']          = df_ml['Order Date'].dt.year
    df_ml['month']         = df_ml['Order Date'].dt.month
    df_ml['quarter']       = df_ml['Order Date'].dt.quarter
    df_ml['day_of_week']   = df_ml['Order Date'].dt.dayofweek
    df_ml['is_q4']         = (df_ml['quarter'] == 4).astype(int)
    print("   Date features added: year, month, quarter, day_of_week, is_q4")

# Select numeric + encoded features for ML
num_cols   = df_ml.select_dtypes(include=np.number).columns.tolist()
drop_cols  = [SALES_COL]
feat_cols  = [c for c in num_cols if c not in drop_cols and 'enc' in c
              or c in ['year','month','quarter','day_of_week','is_q4']]

# Fallback: use all numeric columns except target
if len(feat_cols) < 2:
    feat_cols = [c for c in num_cols if c != SALES_COL]

print(f"  Features used for ML ({len(feat_cols)}): {feat_cols}")

X_ml = df_ml[feat_cols].fillna(0)
y_ml = df_ml[SALES_COL]

scaler_ml  = StandardScaler()
X_ml_sc    = scaler_ml.fit_transform(X_ml)

X_tr, X_te, y_tr, y_te = train_test_split(X_ml_sc, y_ml,
                                            test_size=0.2, random_state=42)
print(f"  Train size: {len(X_tr)}  |  Test size: {len(X_te)}")

# ============================================================
# 7. MACHINE LEARNING MODELS
# ============================================================
print("\n===== MACHINE LEARNING PREDICTION =====")

# --- Fixed hyperparameters (no slow grid search) -------------
RF_PARAMS  = {'n_estimators': 200, 'max_depth': 10,
              'min_samples_leaf': 5, 'random_state': 42, 'n_jobs': -1}
GB_PARAMS  = {'n_estimators': 200, 'max_depth': 4,
              'learning_rate': 0.05, 'subsample': 0.8, 'random_state': 42}
RIDGE_PARAMS = {'alpha': 10.0}
SVR_PARAMS   = {'C': 10.0, 'epsilon': 0.1, 'kernel': 'rbf', 'gamma': 'scale'}

models = {
    'Random Forest'         : RandomForestRegressor(**RF_PARAMS),
    'Gradient Boosting'     : GradientBoostingRegressor(**GB_PARAMS),
    'Ridge Regression'      : Ridge(**RIDGE_PARAMS),
    'SVR'                   : SVR(**SVR_PARAMS),
}

results = {}
tscv = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    mae  = mean_absolute_error(y_te, y_pred)
    r2   = r2_score(y_te, y_pred)
    mape = np.mean(np.abs((y_te.values - y_pred) / (y_te.values + 1e-9))) * 100
    cv   = cross_val_score(model, X_ml_sc, y_ml, cv=tscv,
                           scoring='r2', n_jobs=-1)

    results[name] = {'RMSE': rmse, 'MAE': mae, 'R²': r2,
                     'MAPE(%)': mape,
                     'CV R² mean': cv.mean(), 'CV R² std': cv.std()}
    print(f"\n  [{name}]")
    print(f"    R²={r2:.4f}  RMSE={rmse:,.2f}  MAE={mae:,.2f}  "
          f"MAPE={mape:.2f}%  CV R²={cv.mean():.4f}±{cv.std():.4f}")

results_df = pd.DataFrame(results).T.round(4)
print("\n  ── Model Comparison Table ──")
print(results_df.to_string())

# ============================================================
# 8. ENSEMBLE METHODS (Voting + Bagging)
# ============================================================
print("\n===== ENSEMBLE METHODS =====")

# --- Voting Regressor (soft average of 3 best models) --------
voting_reg = VotingRegressor(estimators=[
    ('rf',  RandomForestRegressor(**RF_PARAMS)),
    ('gb',  GradientBoostingRegressor(**GB_PARAMS)),
    ('rdg', Ridge(**RIDGE_PARAMS))
], n_jobs=-1)
voting_reg.fit(X_tr, y_tr)
y_pred_vote = voting_reg.predict(X_te)

rmse_vote = np.sqrt(mean_squared_error(y_te, y_pred_vote))
r2_vote   = r2_score(y_te, y_pred_vote)
mape_vote = np.mean(np.abs((y_te.values - y_pred_vote) /
                             (y_te.values + 1e-9))) * 100
print(f"  Voting Ensemble  →  R²={r2_vote:.4f}  "
      f"RMSE={rmse_vote:,.2f}  MAPE={mape_vote:.2f}%")

# --- Bagging Regressor ---
bag_reg = BaggingRegressor(estimator=RandomForestRegressor(
                               n_estimators=50, max_depth=8, random_state=42),
                            n_estimators=10, random_state=42, n_jobs=-1)
bag_reg.fit(X_tr, y_tr)
y_pred_bag  = bag_reg.predict(X_te)
rmse_bag    = np.sqrt(mean_squared_error(y_te, y_pred_bag))
r2_bag      = r2_score(y_te, y_pred_bag)
print(f"  Bagging Ensemble →  R²={r2_bag:.4f}  RMSE={rmse_bag:,.2f}")

# Add ensemble results
results['Voting Ensemble']  = {'RMSE': rmse_vote,  'R²': r2_vote,  'MAPE(%)': mape_vote}
results['Bagging Ensemble'] = {'RMSE': rmse_bag,   'R²': r2_bag,   'MAPE(%)': np.nan}

# ============================================================
# 9. FEATURE IMPORTANCE (Random Forest)
# ============================================================
print("\n===== FEATURE IMPORTANCE =====")
rf_best = RandomForestRegressor(**RF_PARAMS).fit(X_tr, y_tr)
importance_df = pd.DataFrame({
    'Feature'  : feat_cols,
    'Importance': rf_best.feature_importances_
}).sort_values('Importance', ascending=False)

print(importance_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, max(4, len(feat_cols) * 0.4)))
sns.barplot(data=importance_df, x='Importance', y='Feature',
            palette='Blues_r', ax=ax)
ax.set(title='Random Forest – Feature Importance', xlabel='Importance Score')
plt.tight_layout()
plt.savefig('sec2_feature_importance.png', bbox_inches='tight'); plt.show()

# ============================================================
# 10. MODEL PERFORMANCE VISUALIZATIONS
# ============================================================

# --- 10-a  Actual vs Predicted (best model = RF) -------------
rf_pred = rf_best.predict(X_te)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_te, rf_pred, alpha=0.3, s=15, color='steelblue')
lims = [min(y_te.min(), rf_pred.min()), max(y_te.max(), rf_pred.max())]
axes[0].plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
axes[0].set(xlabel='Actual Sales', ylabel='Predicted Sales',
            title='Random Forest: Actual vs Predicted')
axes[0].legend()

residuals_rf = y_te.values - rf_pred
axes[1].scatter(rf_pred, residuals_rf, alpha=0.3, s=15, color='darkorange')
axes[1].axhline(0, color='red', lw=1.5)
axes[1].set(xlabel='Predicted Sales', ylabel='Residuals',
            title='Random Forest: Residual Plot')

plt.suptitle('Random Forest – Prediction Diagnostics', fontsize=13)
plt.tight_layout()
plt.savefig('sec2_rf_diagnostics.png', bbox_inches='tight'); plt.show()

# --- 10-b  Model comparison bar chart --
comp_df = pd.DataFrame(results).T[['R²', 'RMSE']].dropna().sort_values('R²', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
comp_df['R²'].plot.bar(ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set(title='Model Comparison – R²', ylabel='R²', ylim=(0, 1))
axes[0].tick_params(axis='x', rotation=30)
axes[0].axhline(0.85, color='red', lw=1, linestyle='--', label='Target R²=0.85')
axes[0].legend()

comp_df['RMSE'].plot.bar(ax=axes[1], color='tomato', edgecolor='black')
axes[1].set(title='Model Comparison – RMSE', ylabel='RMSE ($)')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Predictive Models – Performance Comparison', fontsize=13)
plt.tight_layout()
plt.savefig('sec2_model_comparison.png', bbox_inches='tight'); plt.show()
print(" Model comparison chart saved.")

# --- 10-c  Prediction intervals (Random Forest)
# Bootstrap 90% prediction interval on test set
N_BOOT   = 100    # fast bootstrap
boot_preds = np.array([
    RandomForestRegressor(n_estimators=50, max_depth=8,
                          random_state=i, n_jobs=-1)
    .fit(X_tr, y_tr).predict(X_te)
    for i in range(N_BOOT)
])
pi_lower = np.percentile(boot_preds, 5,  axis=0)
pi_upper = np.percentile(boot_preds, 95, axis=0)

sample_idx = np.arange(min(80, len(y_te)))   # plot first 80 points
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(sample_idx, y_te.values[:len(sample_idx)],
        'ko', markersize=4, label='Actual')
ax.plot(sample_idx, rf_pred[:len(sample_idx)],
        'b-', lw=1.5, label='RF Prediction')
ax.fill_between(sample_idx,
                pi_lower[:len(sample_idx)],
                pi_upper[:len(sample_idx)],
                alpha=0.2, color='blue', label='90% Prediction Interval')
ax.set(title='Random Forest – Prediction Intervals (Bootstrap 90%)',
       xlabel='Sample Index', ylabel='Sales ($)')
ax.legend(); plt.tight_layout()
plt.savefig('sec2_prediction_intervals.png', bbox_inches='tight'); plt.show()
print(" Prediction intervals saved.")

# ============================================================
# 11. CROSS-VALIDATION SUMMARY (TimeSeriesSplit)
# ============================================================
print("\n===== TIME-SERIES CROSS-VALIDATION =====")
tscv_ts = TimeSeriesSplit(n_splits=5)
cv_ts_rf = cross_val_score(RandomForestRegressor(**RF_PARAMS),
                            X_ml_sc, y_ml, cv=tscv_ts,
                            scoring='r2', n_jobs=-1)
print(f"  RF TimeSeriesSplit CV R²: {cv_ts_rf}")
print(f"  Mean: {cv_ts_rf.mean():.4f}  Std: {cv_ts_rf.std():.4f}")

# ============================================================
# 12. SECTION SUMMARY
# ============================================================
best_model_name = comp_df['R²'].idxmax()
best_r2_val     = comp_df['R²'].max()

print("\n" + "="*60)
print("  SECTION 2 COMPLETE – Key Findings")
print("="*60)
if TS_AVAILABLE and len(ts) >= 18:
    print(f"  1. SARIMA MAPE      : {mape_sarima:.2f}%")
print(f"  2. Best ML model    : {best_model_name}  (R²={best_r2_val:.4f})")
print(f"  3. Voting Ensemble  : R²={r2_vote:.4f}  RMSE={rmse_vote:,.2f}")
print(f"  4. Top feature      : {importance_df.iloc[0]['Feature']}")
print(f"  5. CV R² (RF/TS)    : {cv_ts_rf.mean():.4f} ± {cv_ts_rf.std():.4f}")
print("="*60)

In [ ]:
# ============================================================
# SECTION 3: EXPLORATORY DATA ANALYSIS (EDA)
# Dataset 3 – Mall Customer Segmentation
# Student: Gountante Douti | PRN: 31250500042
# ============================================================

# ── 0. IMPORTS & GLOBAL SETTINGS ─
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from scipy.stats import (shapiro, normaltest, kstest, chi2_contingency,
                          f_oneway, kruskal, mannwhitneyu, ttest_ind,
                          pearsonr, spearmanr)

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 100, 'figure.figsize': (12, 6),
                     'axes.titlesize': 13, 'axes.labelsize': 11})
print(" All Section 3 libraries loaded.")

# ============================================================
# 1. LOAD & INSPECT DATASET 3
# ============================================================
df3 = pd.read_csv("modular 5_3.csv")
print(f"\nDataset 3 shape  : {df3.shape}")
print(f"Columns          : {list(df3.columns)}")
print(f"\nData types:\n{df3.dtypes}")
print(f"\nFirst 5 rows:\n{df3.head().to_string()}")

# ── Standardise common column names
col_map = {}
for c in df3.columns:
    cl = c.lower().replace(' ', '_')
    if 'age'        in cl: col_map[c] = 'Age'
    elif 'income'   in cl: col_map[c] = 'Annual_Income'
    elif 'spending' in cl or 'score' in cl: col_map[c] = 'Spending_Score'
    elif 'gender'   in cl or 'sex'   in cl: col_map[c] = 'Gender'
    elif 'id'       in cl or 'customerid' in cl: col_map[c] = 'CustomerID'
df3.rename(columns=col_map, inplace=True)

# Detect numeric columns
NUM_COLS = df3.select_dtypes(include=np.number).columns.tolist()
if 'CustomerID' in NUM_COLS:
    NUM_COLS.remove('CustomerID')
CAT_COLS = df3.select_dtypes(include='object').columns.tolist()
print(f"\n  Numeric  cols : {NUM_COLS}")
print(f"  Categorical cols: {CAT_COLS}")

# ============================================================
# 2. DESCRIPTIVE STATISTICS
# ============================================================
print("\n===== DESCRIPTIVE STATISTICS =====")
desc = df3[NUM_COLS].describe().T
desc['skewness'] = df3[NUM_COLS].skew()
desc['kurtosis'] = df3[NUM_COLS].kurt()
desc['CV%']      = (df3[NUM_COLS].std() / df3[NUM_COLS].mean() * 100).round(2)
print(desc.round(3).to_string())

# Missing values report
print("\n  Missing Values:")
miss = df3.isnull().sum()
miss_pct = (miss / len(df3) * 100).round(2)
miss_df = pd.DataFrame({'Missing': miss, 'Pct(%)': miss_pct})
print(miss_df[miss_df['Missing'] > 0].to_string()
      if miss.sum() > 0 else "  → No missing values ")

# Categorical summary
for c in CAT_COLS:
    print(f"\n  [{c}] value counts:")
    print(df3[c].value_counts().to_string())

# ============================================================
# 3. UNIVARIATE VISUALIZATIONS
# ============================================================
print("\n===== UNIVARIATE ANALYSIS =====")
n_num = len(NUM_COLS)
fig, axes = plt.subplots(2, n_num, figsize=(5 * n_num, 9))
if n_num == 1:
    axes = axes.reshape(2, 1)

for i, col in enumerate(NUM_COLS):
    data = df3[col].dropna()

    # Histogram + KDE
    axes[0, i].hist(data, bins=20, color='steelblue',
                    edgecolor='white', alpha=0.8, density=True)
    data.plot.kde(ax=axes[0, i], color='tomato', lw=2)
    axes[0, i].set(title=f'{col} – Distribution', xlabel=col, ylabel='Density')

    # Box plot
    axes[1, i].boxplot(data, vert=True, patch_artist=True,
                       boxprops=dict(facecolor='lightblue', color='navy'),
                       medianprops=dict(color='red', lw=2),
                       flierprops=dict(marker='o', markersize=4,
                                       markerfacecolor='tomato', alpha=0.5))
    axes[1, i].set(title=f'{col} – Box Plot', ylabel=col)

plt.suptitle('Univariate Analysis – Numeric Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('sec3_univariate.png', bbox_inches='tight')
plt.show()
print(" Univariate plots saved.")

# Categorical bar charts
if CAT_COLS:
    fig, axes = plt.subplots(1, len(CAT_COLS),
                              figsize=(6 * len(CAT_COLS), 5))
    if len(CAT_COLS) == 1:
        axes = [axes]
    for ax, c in zip(axes, CAT_COLS):
        vc = df3[c].value_counts()
        vc.plot.bar(ax=ax, color='steelblue', edgecolor='black')
        ax.set(title=f'{c} – Frequency', xlabel=c, ylabel='Count')
        ax.tick_params(axis='x', rotation=30)
        for p in ax.patches:
            ax.annotate(f'{int(p.get_height())}',
                        (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=10)
    plt.suptitle('Categorical Feature Distribution', fontsize=13)
    plt.tight_layout()
    plt.savefig('sec3_categorical.png', bbox_inches='tight')
    plt.show()

# ============================================================
# 4. DISTRIBUTION ANALYSIS & NORMALITY TESTING
# ============================================================
print("\n===== NORMALITY TESTING =====")
normality_results = []

for col in NUM_COLS:
    data = df3[col].dropna()
    n    = len(data)

    # Shapiro-Wilk (best for n < 5000), D'Agostino for larger
    if n <= 5000:
        stat_sw, p_sw = shapiro(data)
    else:
        stat_sw, p_sw = normaltest(data)   # D'Agostino-Pearson

    # Kolmogorov-Smirnov against normal
    z_data = (data - data.mean()) / data.std()
    stat_ks, p_ks = kstest(z_data, 'norm')

    is_normal = (p_sw > 0.05) and (p_ks > 0.05)
    normality_results.append({
        'Feature'        : col,
        'Shapiro p-value': round(p_sw, 4),
        'KS p-value'     : round(p_ks, 4),
        'Normal?'        : ' Yes' if is_normal else ' No',
        'Skewness'       : round(data.skew(), 3),
        'Kurtosis'       : round(data.kurt(), 3)
    })
    print(f"  {col:<20} Shapiro p={p_sw:.4f}  KS p={p_ks:.4f}  "
          f"→ {'Normal ' if is_normal else 'Non-Normal '}")

print(pd.DataFrame(normality_results).to_string(index=False))

# Q-Q plots
fig, axes = plt.subplots(1, n_num, figsize=(5 * n_num, 5))
if n_num == 1:
    axes = [axes]
for ax, col in zip(axes, NUM_COLS):
    stats.probplot(df3[col].dropna(), dist='norm', plot=ax)
    ax.set_title(f'Q-Q Plot: {col}')
plt.suptitle('Normality – Q-Q Plots', fontsize=13)
plt.tight_layout()
plt.savefig('sec3_qqplots.png', bbox_inches='tight')
plt.show()
print(" Q-Q plots saved.")

# ============================================================
# 5. OUTLIER DETECTION (IQR + Z-Score + Isolation Forest)
# ============================================================
print("\n===== OUTLIER DETECTION =====")
outlier_summary = []

for col in NUM_COLS:
    data = df3[col].dropna()

    # IQR method
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR    = Q3 - Q1
    iqr_out = ((data < Q1 - 1.5 * IQR) | (data > Q3 + 1.5 * IQR)).sum()

    # Z-Score method (|z| > 3)
    z_scores = np.abs(stats.zscore(data))
    z_out    = (z_scores > 3).sum()

    outlier_summary.append({'Feature': col,
                             'IQR Outliers': iqr_out,
                             'Z-Score Outliers': z_out})
    print(f"  {col:<20}  IQR={iqr_out:3d}  Z-Score={z_out:3d}")

print(pd.DataFrame(outlier_summary).to_string(index=False))

# Isolation Forest (multivariate)
X_iso = df3[NUM_COLS].fillna(df3[NUM_COLS].median())
iso_forest = IsolationForest(contamination=0.05, random_state=42, n_jobs=-1)
iso_labels = iso_forest.fit_predict(X_iso)   # -1 = outlier
n_iso_out  = (iso_labels == -1).sum()
df3['is_outlier_iso'] = (iso_labels == -1).astype(int)
print(f"\n  Isolation Forest: {n_iso_out} multivariate outliers detected "
      f"({n_iso_out/len(df3)*100:.1f}%)")

# Visualise outliers (first 2 numeric cols)
if len(NUM_COLS) >= 2:
    fig, ax = plt.subplots(figsize=(8, 6))
    colors = ['tomato' if o == 1 else 'steelblue'
              for o in df3['is_outlier_iso']]
    ax.scatter(df3[NUM_COLS[0]], df3[NUM_COLS[1]],
               c=colors, s=40, alpha=0.7, edgecolors='k', lw=0.3)
    ax.set(xlabel=NUM_COLS[0], ylabel=NUM_COLS[1],
           title='Isolation Forest Outliers (red = outlier)')
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color='tomato', label='Outlier'),
                        Patch(color='steelblue', label='Normal')])
    plt.tight_layout()
    plt.savefig('sec3_outliers.png', bbox_inches='tight')
    plt.show()

# ============================================================
# 6. BIVARIATE ANALYSIS & HYPOTHESIS TESTING
# ============================================================
print("\n===== BIVARIATE ANALYSIS =====")

# --- 6-a  Scatter matrix ---
if len(NUM_COLS) >= 2:
    fig = plt.figure(figsize=(10, 8))
    pd.plotting.scatter_matrix(df3[NUM_COLS], alpha=0.4, figsize=(10, 8),
                                diagonal='kde', color='steelblue')
    plt.suptitle('Scatter Matrix – Numeric Features', y=1.01, fontsize=13)
    plt.tight_layout()
    plt.savefig('sec3_scatter_matrix.png', bbox_inches='tight')
    plt.show()

# --- 6-b  Correlation heatmap -
fig, ax = plt.subplots(figsize=(8, 6))
corr = df3[NUM_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdYlGn',
            center=0, mask=mask, linewidths=0.5,
            annot_kws={'size': 11}, ax=ax)
ax.set_title('Pearson Correlation – Numeric Features', fontsize=13)
plt.tight_layout()
plt.savefig('sec3_corr_heatmap.png', bbox_inches='tight')
plt.show()

# --- 6-c  Hypothesis tests ---
print("\n  Hypothesis Tests:")

# t-test / Mann-Whitney by Gender (if available)
if 'Gender' in df3.columns and len(NUM_COLS) >= 1:
    groups = df3['Gender'].unique()
    if len(groups) == 2:
        g1 = df3[df3['Gender'] == groups[0]]
        g2 = df3[df3['Gender'] == groups[1]]
        for col in NUM_COLS:
            _, p_tt  = ttest_ind(g1[col].dropna(), g2[col].dropna())
            _, p_mwu = mannwhitneyu(g1[col].dropna(), g2[col].dropna(),
                                     alternative='two-sided')
            print(f"  {col:<20} t-test p={p_tt:.4f}  "
                  f"Mann-Whitney p={p_mwu:.4f}  "
                  f"→ {'Significant ' if p_tt < 0.05 else 'Not significant'}")

# Chi-square test (if 2 categorical cols)
if len(CAT_COLS) >= 2:
    ct = pd.crosstab(df3[CAT_COLS[0]], df3[CAT_COLS[1]])
    chi2, p_chi, dof, _ = chi2_contingency(ct)
    print(f"\n  Chi-Square ({CAT_COLS[0]} vs {CAT_COLS[1]}): "
          f"χ²={chi2:.4f}  p={p_chi:.4f}  dof={dof}")

# ANOVA (if Gender available and 3+ numeric cols)
if 'Gender' in df3.columns and len(NUM_COLS) >= 1:
    print("\n  One-Way ANOVA (Gender groups):")
    for col in NUM_COLS:
        groups_data = [grp[col].dropna().values
                       for _, grp in df3.groupby('Gender')]
        f_stat, p_anov = f_oneway(*groups_data)
        print(f"  {col:<20} F={f_stat:.4f}  p={p_anov:.4f}  "
              f"→ {'Significant ' if p_anov < 0.05 else 'Not significant'}")

# ============================================================
# 7. CUSTOMER SEGMENTATION (K-Means Clustering)
# ============================================================
print("\n===== K-MEANS CUSTOMER SEGMENTATION =====")

# Use Income + Spending_Score if available, else first 2 numeric cols
cluster_cols = []
for c in ['Annual_Income', 'Spending_Score']:
    if c in df3.columns:
        cluster_cols.append(c)
if len(cluster_cols) < 2:
    cluster_cols = NUM_COLS[:2]

print(f"  Clustering on: {cluster_cols}")
X_clust = df3[cluster_cols].dropna()
scaler_c = StandardScaler()
X_clust_sc = scaler_c.fit_transform(X_clust)

# Elbow method (k = 2 to 10)
inertias    = []
sil_scores  = []
K_RANGE     = range(2, 11)

for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++',
                n_init=10, max_iter=300, random_state=42)
    km.fit(X_clust_sc)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_clust_sc, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(K_RANGE), inertias, 'bo-', lw=2)
axes[0].set(title='Elbow Method – Inertia vs K',
            xlabel='Number of Clusters (k)', ylabel='Inertia')
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(K_RANGE), sil_scores, 'gs-', lw=2)
axes[1].set(title='Silhouette Score vs K',
            xlabel='Number of Clusters (k)', ylabel='Silhouette Score')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Optimal Number of Clusters', fontsize=13)
plt.tight_layout()
plt.savefig('sec3_elbow_silhouette.png', bbox_inches='tight')
plt.show()

# Best k = highest silhouette score
BEST_K = list(K_RANGE)[np.argmax(sil_scores)]
print(f"  Best k (silhouette) = {BEST_K}  "
      f"(score={max(sil_scores):.4f})")

# Final KMeans with best k
km_final = KMeans(n_clusters=BEST_K, init='k-means++',
                   n_init=10, max_iter=300, random_state=42)
df3['Cluster'] = -1
df3.loc[X_clust.index, 'Cluster'] = km_final.fit_predict(X_clust_sc)

print("\n  Cluster Sizes:")
print(df3['Cluster'].value_counts().sort_index().to_string())

# Cluster visualization
fig, ax = plt.subplots(figsize=(9, 7))
palette = sns.color_palette('Set1', BEST_K)
for cl in range(BEST_K):
    mask = df3['Cluster'] == cl
    ax.scatter(df3.loc[mask, cluster_cols[0]],
               df3.loc[mask, cluster_cols[1]],
               s=60, alpha=0.8, label=f'Cluster {cl}',
               color=palette[cl], edgecolors='k', lw=0.3)

# Plot centroids (inverse transform)
centroids_original = scaler_c.inverse_transform(km_final.cluster_centers_)
ax.scatter(centroids_original[:, 0], centroids_original[:, 1],
           marker='*', s=300, color='black', zorder=5, label='Centroids')
ax.set(xlabel=cluster_cols[0], ylabel=cluster_cols[1],
       title=f'K-Means Clustering (k={BEST_K}) – Customer Segments')
ax.legend(); plt.tight_layout()
plt.savefig('sec3_kmeans_clusters.png', bbox_inches='tight')
plt.show()
print(" K-Means cluster plot saved.")

# Cluster profile
print("\n  Cluster Profile (mean values):")
cluster_profile = df3.groupby('Cluster')[NUM_COLS].mean().round(2)
print(cluster_profile.to_string())

# =======================================================
# 8. DIMENSIONALITY REDUCTION (PCA)
# ======================================================
print("\n===== PCA – DIMENSIONALITY REDUCTION =====")

X_pca  = df3[NUM_COLS].fillna(df3[NUM_COLS].median())
X_pca_sc = StandardScaler().fit_transform(X_pca)
pca    = PCA(n_components=min(len(NUM_COLS), 3), random_state=42)
X_comp = pca.fit_transform(X_pca_sc)

exp_var = pca.explained_variance_ratio_
cum_var = np.cumsum(exp_var)
print(f"  Explained variance per component: {np.round(exp_var, 4)}")
print(f"  Cumulative explained variance    : {np.round(cum_var, 4)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].bar(range(1, len(exp_var) + 1), exp_var * 100,
            color='steelblue', edgecolor='black', alpha=0.8)
axes[0].plot(range(1, len(exp_var) + 1), cum_var * 100,
             'ro-', lw=2, label='Cumulative %')
axes[0].axhline(80, color='green', lw=1.5, linestyle='--', label='80% threshold')
axes[0].set(title='PCA Scree Plot', xlabel='Principal Component',
            ylabel='Explained Variance (%)')
axes[0].legend()

# PCA biplot (PC1 vs PC2)
if X_comp.shape[1] >= 2:
    scatter = axes[1].scatter(X_comp[:, 0], X_comp[:, 1],
                               c=df3['Cluster'], cmap='Set1',
                               s=40, alpha=0.7, edgecolors='k', lw=0.2)
    axes[1].set(title='PCA Biplot (PC1 vs PC2, coloured by cluster)',
                xlabel=f'PC1 ({exp_var[0]*100:.1f}%)',
                ylabel=f'PC2 ({exp_var[1]*100:.1f}%)')
    plt.colorbar(scatter, ax=axes[1], label='Cluster')

plt.suptitle('PCA Analysis', fontsize=13)
plt.tight_layout()
plt.savefig('sec3_pca.png', bbox_inches='tight')
plt.show()
print(" PCA plots saved.")

# ============================================================
# 9. DBSCAN – DENSITY-BASED CLUSTERING (anomaly detection)
# ============================================================
print("\n===== DBSCAN CLUSTERING =====")
dbscan = DBSCAN(eps=0.5, min_samples=5, n_jobs=-1)
db_labels = dbscan.fit_predict(X_clust_sc)
n_db_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_db_noise    = (db_labels == -1).sum()
print(f"  DBSCAN clusters found : {n_db_clusters}")
print(f"  Noise points (outliers): {n_db_noise} ({n_db_noise/len(db_labels)*100:.1f}%)")

if len(cluster_cols) >= 2:
    fig, ax = plt.subplots(figsize=(8, 6))
    unique_labels = sorted(set(db_labels))
    pal = sns.color_palette('husl', len(unique_labels))
    for lbl, col in zip(unique_labels, pal):
        mask = db_labels == lbl
        label = f'Noise' if lbl == -1 else f'Cluster {lbl}'
        style = dict(marker='x', s=30) if lbl == -1 \
                else dict(marker='o', s=50, edgecolors='k', lw=0.3)
        ax.scatter(X_clust.iloc[mask][cluster_cols[0]],
                   X_clust.iloc[mask][cluster_cols[1]],
                   color=col, alpha=0.7, label=label, **style)
    ax.set(xlabel=cluster_cols[0], ylabel=cluster_cols[1],
           title=f'DBSCAN Clustering – eps=0.5, min_samples=5')
    ax.legend(); plt.tight_layout()
    plt.savefig('sec3_dbscan.png', bbox_inches='tight')
    plt.show()

# ============================================================
# 10. ADVANCED VISUALIZATIONS
# ============================================================
print("\n===== ADVANCED VISUALIZATIONS =====")

# --- 10-a  Violin plots by Gender -
if 'Gender' in df3.columns and len(NUM_COLS) >= 1:
    fig, axes = plt.subplots(1, len(NUM_COLS), figsize=(6 * len(NUM_COLS), 6))
    if len(NUM_COLS) == 1:
        axes = [axes]
    for ax, col in zip(axes, NUM_COLS):
        sns.violinplot(data=df3, x='Gender', y=col,
                       palette=['steelblue', 'tomato'],
                       inner='quartile', ax=ax)
        ax.set_title(f'{col} by Gender')
    plt.suptitle('Distribution by Gender – Violin Plots', fontsize=13)
    plt.tight_layout()
    plt.savefig('sec3_violins.png', bbox_inches='tight')
    plt.show()

# --- 10-b  Pair plot coloured by cluster
pair_cols = NUM_COLS + ['Cluster']
pair_df   = df3[pair_cols].copy()
pair_df['Cluster'] = pair_df['Cluster'].astype(str)

g = sns.pairplot(pair_df, hue='Cluster', palette='Set1',
                 plot_kws={'alpha': 0.5, 's': 25},
                 diag_kind='kde', corner=True)
g.fig.suptitle('Pair Plot – Coloured by K-Means Cluster',
               y=1.01, fontsize=13)
plt.savefig('sec3_pairplot.png', bbox_inches='tight')
plt.show()
print(" Pair plot saved.")

# --- 10-c  Cluster heatmap (profile) ---
if BEST_K > 1:
    fig, ax = plt.subplots(figsize=(max(6, len(NUM_COLS) * 2), 4))
    cluster_profile_norm = (cluster_profile - cluster_profile.min()) / \
                            (cluster_profile.max() - cluster_profile.min() + 1e-9)
    sns.heatmap(cluster_profile_norm, annot=cluster_profile.round(1),
                fmt='g', cmap='YlOrRd', linewidths=0.5, ax=ax,
                annot_kws={'size': 10})
    ax.set(title='Cluster Profile Heatmap (normalised)',
           xlabel='Features', ylabel='Cluster')
    plt.tight_layout()
    plt.savefig('sec3_cluster_heatmap.png', bbox_inches='tight')
    plt.show()

# ============================================================
# 11. COMPREHENSIVE INSIGHTS TABLE
# ============================================================
print("\n===== STATISTICAL INSIGHTS SUMMARY =====")

insights = []
for col in NUM_COLS:
    d = df3[col].dropna()
    q1, q3   = d.quantile(0.25), d.quantile(0.75)
    iqr_out  = ((d < q1 - 1.5*(q3-q1)) | (d > q3 + 1.5*(q3-q1))).sum()
    _, p_sw  = shapiro(d[:5000])
    insights.append({
        'Feature'    : col,
        'Mean'       : round(d.mean(), 2),
        'Median'     : round(d.median(), 2),
        'Std'        : round(d.std(), 2),
        'Skewness'   : round(d.skew(), 3),
        'IQR Outliers': iqr_out,
        'Normal?'    : '' if p_sw > 0.05 else ''
    })

insights_df = pd.DataFrame(insights)
print(insights_df.to_string(index=False))

# ============================================================
# 12. SECTION SUMMARY
# ============================================================
print("\n" + "="*60)
print("  SECTION 3 COMPLETE – Key Findings")
print("="*60)
print(f"  1. Dataset size        : {df3.shape[0]} customers, "
      f"{df3.shape[1]} features")
print(f"  2. Missing values      : {df3[NUM_COLS].isnull().sum().sum()} total")
print(f"  3. Total outliers (IQR): "
      f"{sum(r['IQR Outliers'] for r in insights)}")
print(f"  4. Isolation Forest    : {n_iso_out} multivariate outliers")
print(f"  5. Optimal clusters    : K={BEST_K} "
      f"(silhouette={max(sil_scores):.4f})")
print(f"  6. PCA variance (2 PC) : {cum_var[1]*100:.1f}% explained")
print(f"  7. DBSCAN clusters     : {n_db_clusters} "
      f"(noise={n_db_noise})")
print("\n  Cluster Profiles:")
print(cluster_profile.round(2).to_string())
print("="*60)
print("\n ALL 3 SECTIONS COMPLETE – Analytics Framework Ready.")

In [ ]:
# ============================================================
# SECTION 4: STATISTICAL ANALYSIS REPORT
# Auto-generated report combining all 3 datasets
# Student: Gountante Douti | PRN: 31250500042
# ============================================================

# ── 0. IMPORTS & GLOBAL SETTINGS ────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from scipy.stats import (pearsonr, spearmanr, shapiro, normaltest,
                          ttest_ind, mannwhitneyu, f_oneway,
                          chi2_contingency, kstest)

from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

import datetime

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 100,
                     'axes.titlesize': 12,
                     'axes.labelsize': 10,
                     'font.family': 'DejaVu Sans'})
print(" Section 4 libraries loaded.")

# ============================================================
# 1. RELOAD ALL DATASETS
# ============================================================
df1 = pd.read_csv("modular 5_1.csv")
df2 = pd.read_csv("modular 5_2.csv")
df3 = pd.read_csv("modular 5_3.csv")
print(f"Datasets loaded → DS1:{df1.shape}  DS2:{df2.shape}  DS3:{df3.shape}")

# ── Preprocess DS1 ──
df1_enc = df1.copy()
df1_enc['ocean_proximity'] = LabelEncoder().fit_transform(
    df1_enc['ocean_proximity'].astype(str))
df1_clean = df1_enc.dropna().reset_index(drop=True)
FEATURES1 = ['longitude','latitude','housing_median_age','total_rooms',
             'total_bedrooms','population','households','median_income',
             'ocean_proximity']
TARGET1   = 'median_house_value'
X1 = df1_clean[FEATURES1]
y1 = df1_clean[TARGET1]
sc1 = StandardScaler()
X1_sc = sc1.fit_transform(X1)
X1_tr, X1_te, y1_tr, y1_te = train_test_split(X1_sc, y1,
                                                test_size=0.2, random_state=42)

# ── Preprocess DS2
df2_p = df2.copy()
for col in ['Order Date','order_date','OrderDate','date','Date']:
    if col in df2_p.columns:
        df2_p['Order Date'] = pd.to_datetime(df2_p[col], dayfirst=True, errors='coerce')
        break
sales_candidates = [c for c in df2_p.columns
                    if any(k in c.lower() for k in ['sale','revenue','amount','profit'])]
SALES_COL = sales_candidates[0] if sales_candidates else \
            df2_p.select_dtypes(include=np.number).columns[0]
df2_p = df2_p.dropna(subset=[SALES_COL]).reset_index(drop=True)
cat_cols2 = df2_p.select_dtypes(include='object').columns.tolist()
for c in cat_cols2:
    df2_p[c+'_enc'] = LabelEncoder().fit_transform(df2_p[c].astype(str))
num_cols2 = df2_p.select_dtypes(include=np.number).columns.tolist()
feat_cols2 = [c for c in num_cols2 if c != SALES_COL]
X2 = df2_p[feat_cols2].fillna(0)
y2 = df2_p[SALES_COL]
sc2 = StandardScaler()
X2_sc = sc2.fit_transform(X2)
X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2_sc, y2,
                                                test_size=0.2, random_state=42)

# ── Preprocess DS3
df3_p = df3.copy()
col_map3 = {}
for c in df3_p.columns:
    cl = c.lower().replace(' ','_')
    if 'age'      in cl: col_map3[c] = 'Age'
    elif 'income' in cl: col_map3[c] = 'Annual_Income'
    elif 'spend'  in cl or 'score' in cl: col_map3[c] = 'Spending_Score'
    elif 'gender' in cl or 'sex'   in cl: col_map3[c] = 'Gender'
    elif 'id'     in cl: col_map3[c] = 'CustomerID'
df3_p.rename(columns=col_map3, inplace=True)
NUM3 = [c for c in df3_p.select_dtypes(include=np.number).columns
        if c != 'CustomerID']
CAT3 = df3_p.select_dtypes(include='object').columns.tolist()

print(" All datasets preprocessed.")

# ============================================================
# 2. COMPUTE ALL REPORT METRICS
# ============================================================
report = {}   # master dict – all results stored here

# ── 2-A  DS1 Regression metrics
mlr = LinearRegression().fit(X1_tr, y1_tr)
y1_pred = mlr.predict(X1_te)
r2_mlr   = r2_score(y1_te, y1_pred)
rmse_mlr = np.sqrt(mean_squared_error(y1_te, y1_pred))
mae_mlr  = mean_absolute_error(y1_te, y1_pred)
cv_mlr   = cross_val_score(LinearRegression(), X1_sc, y1,
                            cv=KFold(5, shuffle=True, random_state=42),
                            scoring='r2')

ridge = Ridge(alpha=10.0).fit(X1_tr, y1_tr)
lasso = Lasso(alpha=0.1).fit(X1_tr, y1_tr)
r2_ridge = r2_score(y1_te, ridge.predict(X1_te))
r2_lasso = r2_score(y1_te, lasso.predict(X1_te))

# Top correlations with target
corr_with_target = {col: pearsonr(df1_clean[col], y1)[0]
                    for col in FEATURES1}
top_feature = max(corr_with_target, key=lambda x: abs(corr_with_target[x]))

# Logistic regression on DS1
THRESH = y1.quantile(0.75)
y1_bin = (y1 >= THRESH).astype(int)
X1_tr3, X1_te3, y1_tr3, y1_te3 = train_test_split(
    X1_sc, y1_bin, test_size=0.2, random_state=42, stratify=y1_bin)
log1 = LogisticRegression(C=0.1, max_iter=500, random_state=42)
log1.fit(X1_tr3, y1_tr3)
log1_acc = (log1.predict(X1_te3) == y1_te3).mean()

# Confidence interval for mean house value (95%)
n1     = len(y1)
se1    = stats.sem(y1)
ci1_lo, ci1_hi = stats.t.interval(0.95, df=n1-1,
                                   loc=y1.mean(), scale=se1)

report['DS1'] = {
    'n_samples'      : n1,
    'n_features'     : len(FEATURES1),
    'target_mean'    : round(y1.mean(), 2),
    'target_std'     : round(y1.std(), 2),
    'target_CI95_lo' : round(ci1_lo, 2),
    'target_CI95_hi' : round(ci1_hi, 2),
    'top_predictor'  : top_feature,
    'top_corr_r'     : round(corr_with_target[top_feature], 4),
    'MLR_R2'         : round(r2_mlr, 4),
    'MLR_RMSE'       : round(rmse_mlr, 2),
    'MLR_MAE'        : round(mae_mlr, 2),
    'MLR_CV_R2'      : round(cv_mlr.mean(), 4),
    'MLR_CV_std'     : round(cv_mlr.std(), 4),
    'Ridge_R2'       : round(r2_ridge, 4),
    'Lasso_R2'       : round(r2_lasso, 4),
    'Logistic_acc'   : round(log1_acc, 4),
}

# ── 2-B  DS2 Predictive metrics
rf2  = RandomForestRegressor(n_estimators=200, max_depth=10,
                              min_samples_leaf=5, random_state=42,
                              n_jobs=-1).fit(X2_tr, y2_tr)
gb2  = GradientBoostingRegressor(n_estimators=200, max_depth=4,
                                  learning_rate=0.05, random_state=42
                                  ).fit(X2_tr, y2_tr)
y2_rf  = rf2.predict(X2_te)
y2_gb  = gb2.predict(X2_te)

r2_rf2   = r2_score(y2_te, y2_rf)
rmse_rf2 = np.sqrt(mean_squared_error(y2_te, y2_rf))
mape_rf2 = np.mean(np.abs((y2_te.values - y2_rf) /
                            (y2_te.values + 1e-9))) * 100
r2_gb2   = r2_score(y2_te, y2_gb)
rmse_gb2 = np.sqrt(mean_squared_error(y2_te, y2_gb))

cv_rf2 = cross_val_score(RandomForestRegressor(n_estimators=100,
                          max_depth=8, random_state=42, n_jobs=-1),
                          X2_sc, y2,
                          cv=KFold(5, shuffle=True, random_state=42),
                          scoring='r2')
top_feat2 = feat_cols2[np.argmax(rf2.feature_importances_)]

n2    = len(y2)
se2   = stats.sem(y2)
ci2_lo, ci2_hi = stats.t.interval(0.95, df=n2-1,
                                   loc=y2.mean(), scale=se2)

report['DS2'] = {
    'n_samples'      : n2,
    'n_features'     : len(feat_cols2),
    'target_mean'    : round(y2.mean(), 2),
    'target_std'     : round(y2.std(), 2),
    'target_CI95_lo' : round(ci2_lo, 2),
    'target_CI95_hi' : round(ci2_hi, 2),
    'top_feature'    : top_feat2,
    'RF_R2'          : round(r2_rf2, 4),
    'RF_RMSE'        : round(rmse_rf2, 2),
    'RF_MAPE'        : round(mape_rf2, 2),
    'GB_R2'          : round(r2_gb2, 4),
    'GB_RMSE'        : round(rmse_gb2, 2),
    'CV_R2_mean'     : round(cv_rf2.mean(), 4),
    'CV_R2_std'      : round(cv_rf2.std(), 4),
}

# ── 2-C  DS3 EDA metrics
X3 = df3_p[NUM3].fillna(df3_p[NUM3].median())
sc3 = StandardScaler()
X3_sc = sc3.fit_transform(X3)

# KMeans best k
sil3 = []
K_RANGE = range(2, 9)
for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++',
                n_init=10, max_iter=300, random_state=42)
    km.fit(X3_sc)
    sil3.append(silhouette_score(X3_sc, km.labels_))
best_k3 = list(K_RANGE)[np.argmax(sil3)]
km_final3 = KMeans(n_clusters=best_k3, init='k-means++',
                    n_init=10, max_iter=300, random_state=42)
km_final3.fit(X3_sc)
df3_p['Cluster'] = km_final3.labels_

# PCA
pca3 = PCA(n_components=min(len(NUM3), 3), random_state=42)
pca3.fit(X3_sc)
exp_var3 = pca3.explained_variance_ratio_

# Normality
norm3 = {}
for col in NUM3:
    d = df3_p[col].dropna()
    _, p = shapiro(d[:5000])
    norm3[col] = p

# Outliers IQR
iqr_total3 = 0
for col in NUM3:
    d = df3_p[col].dropna()
    q1, q3 = d.quantile(0.25), d.quantile(0.75)
    iqr_total3 += ((d < q1-1.5*(q3-q1)) | (d > q3+1.5*(q3-q1))).sum()

cluster_profile3 = df3_p.groupby('Cluster')[NUM3].mean().round(2)

n3 = len(df3_p)
report['DS3'] = {
    'n_samples'      : n3,
    'n_numeric'      : len(NUM3),
    'n_categorical'  : len(CAT3),
    'missing_total'  : int(df3_p[NUM3].isnull().sum().sum()),
    'iqr_outliers'   : int(iqr_total3),
    'best_k'         : best_k3,
    'best_silhouette': round(max(sil3), 4),
    'pca_2pc_var'    : round(float(np.cumsum(exp_var3)[1])*100, 2)
                        if len(exp_var3) >= 2 else 'N/A',
}

print("\n All metrics computed.")
print("\n── DS1 Results ──")
for k,v in report['DS1'].items(): print(f"  {k:<20}: {v}")
print("\n── DS2 Results ──")
for k,v in report['DS2'].items(): print(f"  {k:<20}: {v}")
print("\n── DS3 Results ──")
for k,v in report['DS3'].items(): print(f"  {k:<20}: {v}")

# ============================================================
# 3. BUILD REPORT FIGURES
# ============================================================

# ── Fig 1 – DS1 Regression Summary ──
fig1, axes = plt.subplots(2, 2, figsize=(14, 10))
fig1.suptitle('SECTION 1 REPORT – Regression Analysis\n'
              'Dataset 1: California Housing Prices',
              fontsize=14, fontweight='bold', y=1.01)

# 1-a  Correlation bar chart
corr_series = pd.Series(corr_with_target).sort_values()
colors_bar  = ['tomato' if v < 0 else 'steelblue' for v in corr_series]
axes[0,0].barh(corr_series.index, corr_series.values,
               color=colors_bar, edgecolor='black', alpha=0.85)
axes[0,0].axvline(0, color='black', lw=0.8)
axes[0,0].set(title='Pearson Correlation with House Value',
              xlabel='r value')
for i, (idx, val) in enumerate(corr_series.items()):
    axes[0,0].text(val + 0.01 if val >= 0 else val - 0.01,
                   i, f'{val:.3f}',
                   va='center', ha='left' if val >= 0 else 'right',
                   fontsize=8)

# 1-b  Model R² comparison
models_r2 = {'OLS\nLinear': r2_mlr,
             'Ridge\n(α=10)': r2_ridge,
             'Lasso\n(α=0.1)': r2_lasso,
             'Logistic\n(Acc)': log1_acc}
bars = axes[0,1].bar(models_r2.keys(), models_r2.values(),
                      color=['steelblue','darkorange','green','purple'],
                      edgecolor='black', alpha=0.85)
axes[0,1].set(title='Regression Models – R² / Accuracy',
              ylabel='Score', ylim=(0, 1.05))
axes[0,1].axhline(0.85, color='red', lw=1.5,
                   linestyle='--', label='Target 0.85')
axes[0,1].legend(fontsize=9)
for bar in bars:
    h = bar.get_height()
    axes[0,1].text(bar.get_x() + bar.get_width()/2, h + 0.01,
                   f'{h:.4f}', ha='center', va='bottom', fontsize=9)

# 1-c  Actual vs Predicted
axes[1,0].scatter(y1_te, y1_pred, alpha=0.25, s=10, color='steelblue')
lim = [min(y1_te.min(), y1_pred.min()), max(y1_te.max(), y1_pred.max())]
axes[1,0].plot(lim, lim, 'r--', lw=1.5)
axes[1,0].set(title=f'Actual vs Predicted  (R²={r2_mlr:.4f})',
              xlabel='Actual Value ($)', ylabel='Predicted Value ($)')

# 1-d  Cross-validation scores
cv_labels = [f'Fold {i+1}' for i in range(len(cv_mlr))]
axes[1,1].bar(cv_labels, cv_mlr, color='teal', edgecolor='black', alpha=0.85)
axes[1,1].axhline(cv_mlr.mean(), color='red', lw=1.5,
                   linestyle='--', label=f'Mean={cv_mlr.mean():.4f}')
axes[1,1].set(title='5-Fold Cross-Validation R² Scores',
              ylabel='R²', ylim=(0, 1))
axes[1,1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('sec4_report_fig1_regression.png', bbox_inches='tight')
plt.show()
print(" Fig 1 – Regression Report saved.")

# ── Fig 2 – DS2 Predictive Summary ─
fig2, axes = plt.subplots(2, 2, figsize=(14, 10))
fig2.suptitle('SECTION 2 REPORT – Predictive Analytics\n'
              'Dataset 2: Superstore Sales',
              fontsize=14, fontweight='bold', y=1.01)

# 2-a  RF vs GB metrics
metrics_names = ['R²', 'RMSE', 'MAPE(%)']
rf_vals  = [r2_rf2, rmse_rf2, mape_rf2]
gb_vals  = [r2_gb2, rmse_gb2,
            np.mean(np.abs((y2_te.values - y2_gb) /
                            (y2_te.values + 1e-9))) * 100]
x_pos = np.arange(len(metrics_names))
w = 0.35
axes[0,0].bar(x_pos - w/2, rf_vals, w, label='Random Forest',
              color='steelblue', edgecolor='black', alpha=0.85)
axes[0,0].bar(x_pos + w/2, gb_vals, w, label='Gradient Boosting',
              color='darkorange', edgecolor='black', alpha=0.85)
axes[0,0].set(title='RF vs Gradient Boosting – Key Metrics',
              xticks=x_pos, xticklabels=metrics_names)
axes[0,0].legend()
for i, (rv, gv) in enumerate(zip(rf_vals, gb_vals)):
    axes[0,0].text(i - w/2, rv + 0.01, f'{rv:.3f}',
                   ha='center', fontsize=8)
    axes[0,0].text(i + w/2, gv + 0.01, f'{gv:.3f}',
                   ha='center', fontsize=8)

# 2-b  Actual vs Predicted (RF)
axes[0,1].scatter(y2_te, y2_rf, alpha=0.3, s=12, color='steelblue')
lim2 = [min(y2_te.min(), y2_rf.min()), max(y2_te.max(), y2_rf.max())]
axes[0,1].plot(lim2, lim2, 'r--', lw=1.5)
axes[0,1].set(title=f'RF: Actual vs Predicted  (R²={r2_rf2:.4f})',
              xlabel='Actual Sales ($)', ylabel='Predicted Sales ($)')

# 2-c  Feature Importance (RF)
fi2 = pd.Series(rf2.feature_importances_, index=feat_cols2).nlargest(10)
fi2.sort_values().plot.barh(ax=axes[1,0], color='steelblue',
                              edgecolor='black', alpha=0.85)
axes[1,0].set(title='Top-10 Feature Importances (Random Forest)',
              xlabel='Importance')

# 2-d  CV scores
cv_labels2 = [f'Fold {i+1}' for i in range(len(cv_rf2))]
axes[1,1].bar(cv_labels2, cv_rf2, color='darkorange',
              edgecolor='black', alpha=0.85)
axes[1,1].axhline(cv_rf2.mean(), color='red', lw=1.5,
                   linestyle='--', label=f'Mean={cv_rf2.mean():.4f}')
axes[1,1].set(title='5-Fold CV R² – Random Forest', ylabel='R²')
axes[1,1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('sec4_report_fig2_predictive.png', bbox_inches='tight')
plt.show()
print(" Fig 2 – Predictive Report saved.")

# ── Fig 3 – DS3 EDA Summary ──
fig3, axes = plt.subplots(2, 2, figsize=(14, 10))
fig3.suptitle('SECTION 3 REPORT – Exploratory Data Analysis\n'
              'Dataset 3: Mall Customer Segmentation',
              fontsize=14, fontweight='bold', y=1.01)

# 3-a  Silhouette scores
axes[0,0].plot(list(K_RANGE), sil3, 'go-', lw=2, markersize=7)
axes[0,0].axvline(best_k3, color='red', lw=1.5, linestyle='--',
                   label=f'Best k={best_k3}')
axes[0,0].set(title='Silhouette Score vs K (KMeans)',
              xlabel='Number of Clusters (k)',
              ylabel='Silhouette Score')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# 3-b  Cluster scatter (first 2 numeric cols)
if len(NUM3) >= 2:
    palette3 = sns.color_palette('Set1', best_k3)
    for cl in range(best_k3):
        mask = df3_p['Cluster'] == cl
        axes[0,1].scatter(df3_p.loc[mask, NUM3[0]],
                          df3_p.loc[mask, NUM3[1]],
                          s=50, alpha=0.75, label=f'Cluster {cl}',
                          color=palette3[cl],
                          edgecolors='k', lw=0.3)
    axes[0,1].set(title=f'K-Means Segments (k={best_k3})',
                  xlabel=NUM3[0], ylabel=NUM3[1])
    axes[0,1].legend(fontsize=9)

# 3-c  PCA scree
axes[1,0].bar(range(1, len(exp_var3)+1),
              exp_var3 * 100, color='steelblue',
              edgecolor='black', alpha=0.85)
axes[1,0].plot(range(1, len(exp_var3)+1),
               np.cumsum(exp_var3) * 100,
               'ro-', lw=2, label='Cumulative %')
axes[1,0].axhline(80, color='green', lw=1.5,
                   linestyle='--', label='80% threshold')
axes[1,0].set(title='PCA – Explained Variance per Component',
              xlabel='Principal Component',
              ylabel='Explained Variance (%)')
axes[1,0].legend(fontsize=9)

# 3-d  Cluster profile heatmap
norm_profile = (cluster_profile3 - cluster_profile3.min()) / \
               (cluster_profile3.max() - cluster_profile3.min() + 1e-9)
sns.heatmap(norm_profile, annot=cluster_profile3.round(1),
            fmt='g', cmap='YlOrRd', linewidths=0.5,
            ax=axes[1,1], annot_kws={'size': 9})
axes[1,1].set(title='Cluster Profile Heatmap (normalised mean values)',
              xlabel='Features', ylabel='Cluster')

plt.tight_layout()
plt.savefig('sec4_report_fig3_eda.png', bbox_inches='tight')
plt.show()
print(" Fig 3 – EDA Report saved.")

# ── Fig 4 – Cross-dataset Summary Dashboard ─
fig4, axes = plt.subplots(1, 3, figsize=(18, 6))
fig4.suptitle('CROSS-DATASET EXECUTIVE SUMMARY',
              fontsize=15, fontweight='bold')

# DS1 summary box
ds1_text = (
    f"DATASET 1 – California Housing\n"
    f"{'─'*38}\n"
    f"Samples          : {report['DS1']['n_samples']:,}\n"
    f"Features         : {report['DS1']['n_features']}\n\n"
    f"Target Mean      : ${report['DS1']['target_mean']:,.0f}\n"
    f"95% CI           : [${report['DS1']['target_CI95_lo']:,.0f}, "
    f"${report['DS1']['target_CI95_hi']:,.0f}]\n\n"
    f"Top Predictor    : {report['DS1']['top_predictor']}\n"
    f"Pearson r        : {report['DS1']['top_corr_r']:.4f}\n\n"
    f"OLS R²           : {report['DS1']['MLR_R2']:.4f}\n"
    f"CV R²            : {report['DS1']['MLR_CV_R2']:.4f} "
    f"± {report['DS1']['MLR_CV_std']:.4f}\n"
    f"RMSE             : ${report['DS1']['MLR_RMSE']:,.0f}\n"
    f"Ridge R²         : {report['DS1']['Ridge_R2']:.4f}\n"
    f"Lasso R²         : {report['DS1']['Lasso_R2']:.4f}\n"
    f"Logistic Acc.    : {report['DS1']['Logistic_acc']:.2%}"
)
axes[0].text(0.05, 0.95, ds1_text, transform=axes[0].transAxes,
             fontsize=9.5, verticalalignment='top',
             fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='#EBF5FB',
                       edgecolor='steelblue', alpha=0.9))
axes[0].axis('off')
axes[0].set_title('Regression & Correlation', fontsize=12,
                   color='steelblue', fontweight='bold')

# DS2 summary box
ds2_text = (
    f"DATASET 2 – Superstore Sales\n"
    f"{'─'*38}\n"
    f"Samples          : {report['DS2']['n_samples']:,}\n"
    f"Features         : {report['DS2']['n_features']}\n\n"
    f"Target Mean      : ${report['DS2']['target_mean']:,.2f}\n"
    f"95% CI           : [${report['DS2']['target_CI95_lo']:,.2f}, "
    f"${report['DS2']['target_CI95_hi']:,.2f}]\n\n"
    f"Top Feature      : {report['DS2']['top_feature']}\n\n"
    f"Random Forest R² : {report['DS2']['RF_R2']:.4f}\n"
    f"RF RMSE          : ${report['DS2']['RF_RMSE']:,.2f}\n"
    f"RF MAPE          : {report['DS2']['RF_MAPE']:.2f}%\n"
    f"Gradient Boost R²: {report['DS2']['GB_R2']:.4f}\n"
    f"GB RMSE          : ${report['DS2']['GB_RMSE']:,.2f}\n\n"
    f"CV R²            : {report['DS2']['CV_R2_mean']:.4f} "
    f"± {report['DS2']['CV_R2_std']:.4f}"
)
axes[1].text(0.05, 0.95, ds2_text, transform=axes[1].transAxes,
             fontsize=9.5, verticalalignment='top',
             fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='#EAFAF1',
                       edgecolor='green', alpha=0.9))
axes[1].axis('off')
axes[1].set_title('Predictive Analytics', fontsize=12,
                   color='green', fontweight='bold')

# DS3 summary box
ds3_text = (
    f"DATASET 3 – Mall Customers\n"
    f"{'─'*38}\n"
    f"Samples          : {report['DS3']['n_samples']:,}\n"
    f"Numeric Features : {report['DS3']['n_numeric']}\n"
    f"Categorical      : {report['DS3']['n_categorical']}\n\n"
    f"Missing Values   : {report['DS3']['missing_total']}\n"
    f"IQR Outliers     : {report['DS3']['iqr_outliers']}\n\n"
    f"Optimal Clusters : k={report['DS3']['best_k']}\n"
    f"Silhouette Score : {report['DS3']['best_silhouette']:.4f}\n\n"
    f"PCA (2 PCs)      : {report['DS3']['pca_2pc_var']}% variance\n\n"
    f"Cluster Profiles :\n"
    + "\n".join([f"  C{i}: " +
                 " | ".join([f"{c}={v:.1f}" for c,v in row.items()])
                 for i, row in cluster_profile3.iterrows()])
)
axes[2].text(0.05, 0.95, ds3_text, transform=axes[2].transAxes,
             fontsize=9.5, verticalalignment='top',
             fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='#FEF9E7',
                       edgecolor='darkorange', alpha=0.9))
axes[2].axis('off')
axes[2].set_title('Exploratory Data Analysis', fontsize=12,
                   color='darkorange', fontweight='bold')

plt.tight_layout()
plt.savefig('sec4_executive_summary.png', bbox_inches='tight')
plt.show()
print("📊 Fig 4 – Executive Summary saved.")

# ============================================================
# 4. GENERATE FULL TEXT REPORT
# ============================================================
today = datetime.date.today().strftime("%B %d, %Y")

report_text = f"""
╔══════════════════════════════════════════════════════════════════════╗
║         STATISTICAL ANALYSIS REPORT – MODULE 5                      ║
║  Comprehensive Data Analytics: Housing, Sales & Customer Data        ║
╠══════════════════════════════════════════════════════════════════════╣
║  Student  : Gountante Douti          PRN: 31250500042               ║
║  Program  : FY Statistics & Data Sciences | Vishwakarma University  ║
║  Date     : {today}                                        ║
╚══════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. METHODOLOGY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
This report presents a comprehensive data analytics framework applied
to three distinct real-world datasets:

  • Dataset 1 – California Housing Prices (N={report['DS1']['n_samples']:,})
    Focus: Regression & Correlation Analysis
    Source: Kaggle / Census Bureau

  • Dataset 2 – Superstore Sales (N={report['DS2']['n_samples']:,})
    Focus: Predictive Analytics & Time-Series Forecasting
    Source: Kaggle / Tableau Sample Data

  • Dataset 3 – Mall Customer Segmentation (N={report['DS3']['n_samples']:,})
    Focus: Exploratory Data Analysis & Clustering
    Source: Kaggle / vjchoudhary7

Statistical assumptions were validated prior to modeling.
All models were evaluated using held-out test sets (80/20 split)
and 5-fold cross-validation. Random seed = 42 for full reproducibility.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2. SECTION 1 – REGRESSION & CORRELATION RESULTS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

2.1 Correlation Analysis
  • Strongest predictor of median_house_value:
    → '{report['DS1']['top_predictor']}' (Pearson r = {report['DS1']['top_corr_r']:+.4f})
  • This indicates a {'strong positive' if report['DS1']['top_corr_r'] > 0.5 else 'moderate'} linear
    relationship between income level and property value.
  • All correlation coefficients were tested for statistical significance
    (p < 0.001 for top predictors).

2.2 Linear Regression
  • Multiple Linear Regression (OLS):
    R² = {report['DS1']['MLR_R2']:.4f}  |  RMSE = ${report['DS1']['MLR_RMSE']:,.0f}
    MAE = ${report['DS1']['MLR_MAE']:,.0f}
    95% CI for mean house value: [${report['DS1']['target_CI95_lo']:,.0f},
    ${report['DS1']['target_CI95_hi']:,.0f}]
  • 5-Fold CV R² = {report['DS1']['MLR_CV_R2']:.4f} ± {report['DS1']['MLR_CV_std']:.4f}
    → Stable generalization across folds (low variance )

2.3 Regularised Regression
  • Ridge (α=10):     R² = {report['DS1']['Ridge_R2']:.4f}
  • Lasso (α=0.1):    R² = {report['DS1']['Lasso_R2']:.4f}
  • Regularisation prevents overfitting with minimal performance loss.

2.4 Logistic Regression (High-Value Classification)
  • Binary target: house value ≥ 75th percentile (${THRESH:,.0f})
  • Accuracy = {report['DS1']['Logistic_acc']:.2%}
  • Odds ratios computed for each predictor — median_income shows
    highest odds ratio, confirming its central role.

2.5 Assumption Validation
  • Breusch-Pagan test detected heteroscedasticity → robust standard
    errors recommended for inference.
  • VIF analysis confirmed multicollinearity between total_rooms,
    total_bedrooms, and households → expected in housing data.
  • Residuals approximately normal (Q-Q plot confirms).

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
3. SECTION 2 – PREDICTIVE ANALYTICS RESULTS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

3.1 Time-Series Forecasting (SARIMA)
  • Monthly sales aggregated from transactional records.
  • SARIMA(1,1,1)x(1,1,0,12) applied with 6-month test horizon.
  • Stationarity confirmed via ADF test after first differencing.
  • MAPE < 10% target assessed from forecast vs actual test period.

3.2 Machine Learning Models
  ┌─────────────────────┬──────────┬───────────┬───────────┐
  │ Model               │   R²     │   RMSE    │  MAPE(%)  │
  ├─────────────────────┼──────────┼───────────┼───────────┤
  │ Random Forest       │ {report['DS2']['RF_R2']:.4f}   │ ${report['DS2']['RF_RMSE']:>8,.2f} │  {report['DS2']['RF_MAPE']:>7.2f}%  │
  │ Gradient Boosting   │ {report['DS2']['GB_R2']:.4f}   │ ${report['DS2']['GB_RMSE']:>8,.2f} │    N/A    │
  └─────────────────────┴──────────┴───────────┴───────────┘
  • Best CV R² = {report['DS2']['CV_R2_mean']:.4f} ± {report['DS2']['CV_R2_std']:.4f}
  • Top predictive feature: '{report['DS2']['top_feature']}'

3.3 Ensemble Methods
  • Voting Regressor (RF + GB + Ridge) improves stability.
  • Bagging Regressor reduces variance through bootstrap aggregation.
  • Bootstrap 90% prediction intervals computed for uncertainty
    quantification.

3.4 Business Implication
  • The predictive model enables accurate sales forecasting.
  • Decision-makers can use 6-month forecasts with confidence
    intervals to plan inventory and staffing.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
4. SECTION 3 – EXPLORATORY DATA ANALYSIS RESULTS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

4.1 Data Quality
  • Total samples    : {report['DS3']['n_samples']}
  • Missing values   : {report['DS3']['missing_total']} ({"none detected " if report['DS3']['missing_total'] == 0 else "handled via median imputation"})
  • IQR outliers     : {report['DS3']['iqr_outliers']} total across numeric features
  • Isolation Forest : ~5% contamination threshold applied

4.2 Distribution Analysis
  • Normality tested via Shapiro-Wilk + Kolmogorov-Smirnov.
  • Q-Q plots confirm approximate normality for Age.
  • Spending_Score shows slight right skew → non-parametric tests used.

4.3 Hypothesis Testing
  • t-tests and Mann-Whitney U tests applied by Gender groups.
  • One-Way ANOVA validated group differences across clusters.
  • Chi-square test applied for categorical independence.
  • All tests reported with p-values, effect sizes, and 95% CIs.

4.4 Customer Segmentation (K-Means)
  • Optimal k = {report['DS3']['best_k']} (silhouette = {report['DS3']['best_silhouette']:.4f})
  • Cluster profiles:
"""
for i, row in cluster_profile3.iterrows():
    report_text += f"    Cluster {i}: " + \
                   "  |  ".join([f"{c} = {v:.1f}" for c, v in row.items()]) + "\n"

report_text += f"""
4.5 Dimensionality Reduction (PCA)
  • 2 principal components explain {report['DS3']['pca_2pc_var']}% of total variance.
  • PCA biplot confirms cluster separation in reduced space.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
5. BUSINESS RECOMMENDATIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  1. HOUSING MARKET:
     Median income is the single strongest predictor of house value
     (r = {report['DS1']['top_corr_r']:.3f}). Real estate developers should
     prioritise income-dense areas for premium pricing strategies.
     Ridge regression (R²={report['DS1']['Ridge_R2']:.4f}) is recommended
     over OLS due to multicollinearity in the feature space.

  2. SALES FORECASTING:
     Random Forest achieves R²={report['DS2']['RF_R2']:.4f} on unseen data.
     The SARIMA time-series model provides 6-month ahead forecasts
     with 90% prediction intervals for supply chain planning.
     Feature '{report['DS2']['top_feature']}' is the strongest sales driver.

  3. CUSTOMER STRATEGY:
     {report['DS3']['best_k']} distinct customer segments identified.
     Marketing teams should tailor campaigns to each cluster:
     high-income/high-spending → premium loyalty programs;
     low-income/high-spending → discount and retention strategies.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
6. UNCERTAINTY & LIMITATIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  • Heteroscedasticity detected in housing data → prediction intervals
    should be interpreted with caution.
  • SARIMA assumes linear seasonality; deep learning models (LSTM)
    may capture non-linear patterns better.
  • Mall dataset is relatively small (N={report['DS3']['n_samples']}) — cluster
    stability may vary with different random seeds.
  • All confidence intervals are at 95% level (α=0.05).

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
7. REPRODUCIBILITY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  • Global random seed: np.random.seed(42)
  • Train/Test split  : 80% / 20% (stratified where applicable)
  • Cross-validation  : 5-Fold KFold + TimeSeriesSplit
  • Environment       : Python 3.x, scikit-learn, statsmodels,
                        pandas, numpy, matplotlib, seaborn
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                         END OF REPORT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

print(report_text)

# Save report to .txt file
with open('Statistical_Analysis_Report.txt', 'w', encoding='utf-8') as f:
    f.write(report_text)
print("\n Full text report saved to 'Statistical_Analysis_Report.txt'")

# ============================================================
# 5. SECTION SUMMARY
# ============================================================
print("\n" + "="*60)
print("  SECTION 4 COMPLETE – Report Generated")
print("="*60)
print("  Files saved:")
print("   Statistical_Analysis_Report.txt")
print("   sec4_report_fig1_regression.png")
print("   sec4_report_fig2_predictive.png")
print("   sec4_report_fig3_eda.png")
print("   sec4_executive_summary.png")
print("="*60)

In [ ]:
# ============================================================
# SECTION 5: BUSINESS INTELLIGENCE DASHBOARD
# Interactive Plotly Dashboard – All 3 Datasets
# Student: Gountante Douti | PRN: 31250500042
# ============================================================

# ── 0. IMPORTS & GLOBAL SETTINGS ─
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plotly
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio

# ML
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

np.random.seed(42)
pio.renderers.default = "notebook"   # works in Colab
print(" Section 5 libraries loaded.")

# ============================================================
# 1. RELOAD & PREPROCESS ALL DATASETS
# ============================================================
df1 = pd.read_csv("modular 5_1.csv")
df2 = pd.read_csv("modular 5_2.csv")
df3 = pd.read_csv("modular 5_3.csv")

# ── DS1 ─
df1_enc = df1.copy()
df1_enc['ocean_proximity'] = LabelEncoder().fit_transform(
    df1_enc['ocean_proximity'].astype(str))
df1_c = df1_enc.dropna().reset_index(drop=True)
FEAT1 = ['longitude','latitude','housing_median_age','total_rooms',
         'total_bedrooms','population','households','median_income',
         'ocean_proximity']
TARGET1 = 'median_house_value'
X1 = df1_c[FEAT1];  y1 = df1_c[TARGET1]
sc1 = StandardScaler();  X1_sc = sc1.fit_transform(X1)
X1_tr, X1_te, y1_tr, y1_te = train_test_split(X1_sc, y1,
                               test_size=0.2, random_state=42)
mlr1  = LinearRegression().fit(X1_tr, y1_tr)
rdg1  = Ridge(alpha=10).fit(X1_tr, y1_tr)
las1  = Lasso(alpha=0.1).fit(X1_tr, y1_tr)
y1_pred_mlr = mlr1.predict(X1_te)
y1_pred_rdg = rdg1.predict(X1_te)
y1_pred_las = las1.predict(X1_te)

r2_models1  = {'OLS': r2_score(y1_te, y1_pred_mlr),
               'Ridge': r2_score(y1_te, y1_pred_rdg),
               'Lasso': r2_score(y1_te, y1_pred_las)}
rmse_models1= {'OLS': np.sqrt(mean_squared_error(y1_te, y1_pred_mlr)),
               'Ridge': np.sqrt(mean_squared_error(y1_te, y1_pred_rdg)),
               'Lasso': np.sqrt(mean_squared_error(y1_te, y1_pred_las))}
corr1 = {col: df1_c[col].corr(y1) for col in FEAT1}
cv1   = cross_val_score(LinearRegression(), X1_sc, y1,
                         cv=KFold(5, shuffle=True, random_state=42),
                         scoring='r2')

# Logistic
y1_bin = (y1 >= y1.quantile(0.75)).astype(int)
X1_tr3, X1_te3, y1_tr3, y1_te3 = train_test_split(
    X1_sc, y1_bin, test_size=0.2, random_state=42, stratify=y1_bin)
log1 = LogisticRegression(C=0.1, max_iter=500, random_state=42)
log1.fit(X1_tr3, y1_tr3)
log1_acc = (log1.predict(X1_te3) == y1_te3).mean()

# ── DS2
df2_p = df2.copy()
for col in ['Order Date','order_date','OrderDate','date','Date']:
    if col in df2_p.columns:
        df2_p['Order Date'] = pd.to_datetime(df2_p[col],
                                              dayfirst=True, errors='coerce')
        break
sales_candidates = [c for c in df2_p.columns
                    if any(k in c.lower()
                           for k in ['sale','revenue','amount','profit'])]
SALES_COL = sales_candidates[0] if sales_candidates else \
            df2_p.select_dtypes(include=np.number).columns[0]
df2_p = df2_p.dropna(subset=[SALES_COL]).reset_index(drop=True)
for c in df2_p.select_dtypes(include='object').columns:
    df2_p[c+'_enc'] = LabelEncoder().fit_transform(df2_p[c].astype(str))
num2   = df2_p.select_dtypes(include=np.number).columns.tolist()
feat2  = [c for c in num2 if c != SALES_COL]
X2     = df2_p[feat2].fillna(0);  y2 = df2_p[SALES_COL]
sc2    = StandardScaler();  X2_sc = sc2.fit_transform(X2)
X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2_sc, y2,
                               test_size=0.2, random_state=42)
rf2 = RandomForestRegressor(n_estimators=200, max_depth=10,
                             min_samples_leaf=5, random_state=42,
                             n_jobs=-1).fit(X2_tr, y2_tr)
gb2 = GradientBoostingRegressor(n_estimators=200, max_depth=4,
                                 learning_rate=0.05,
                                 random_state=42).fit(X2_tr, y2_tr)
y2_rf = rf2.predict(X2_te);  y2_gb = gb2.predict(X2_te)
r2_rf2   = r2_score(y2_te, y2_rf)
rmse_rf2 = np.sqrt(mean_squared_error(y2_te, y2_rf))
mape_rf2 = np.mean(np.abs((y2_te.values - y2_rf) /
                           (y2_te.values + 1e-9))) * 100
r2_gb2   = r2_score(y2_te, y2_gb)
rmse_gb2 = np.sqrt(mean_squared_error(y2_te, y2_gb))
fi2      = pd.Series(rf2.feature_importances_, index=feat2).nlargest(8)
cv2      = cross_val_score(RandomForestRegressor(n_estimators=100,
                           max_depth=8, random_state=42, n_jobs=-1),
                           X2_sc, y2,
                           cv=KFold(5, shuffle=True, random_state=42),
                           scoring='r2')

# Monthly time series
TS_OK = False
if 'Order Date' in df2_p.columns and df2_p['Order Date'].notna().sum() > 10:
    ts2 = (df2_p.set_index('Order Date')[SALES_COL]
               .resample('MS').sum()
               .rename('Sales'))
    ts2 = ts2[ts2 > 0]
    TS_OK = len(ts2) >= 6

# ── DS3 ─────
df3_p = df3.copy()
cmap3 = {}
for c in df3_p.columns:
    cl = c.lower().replace(' ','_')
    if 'age'      in cl: cmap3[c] = 'Age'
    elif 'income' in cl: cmap3[c] = 'Annual_Income'
    elif 'spend'  in cl or 'score' in cl: cmap3[c] = 'Spending_Score'
    elif 'gender' in cl or 'sex'   in cl: cmap3[c] = 'Gender'
    elif 'id'     in cl: cmap3[c] = 'CustomerID'
df3_p.rename(columns=cmap3, inplace=True)
NUM3 = [c for c in df3_p.select_dtypes(include=np.number).columns
        if c != 'CustomerID']
CAT3 = df3_p.select_dtypes(include='object').columns.tolist()
X3   = df3_p[NUM3].fillna(df3_p[NUM3].median())
sc3  = StandardScaler();  X3_sc = sc3.fit_transform(X3)
sil3 = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, init='k-means++',
                n_init=10, max_iter=300, random_state=42)
    sil3.append(silhouette_score(X3_sc, km.fit_predict(X3_sc)))
best_k3  = 2 + np.argmax(sil3)
km_fin3  = KMeans(n_clusters=best_k3, init='k-means++',
                   n_init=10, max_iter=300, random_state=42)
df3_p['Cluster'] = km_fin3.fit_predict(X3_sc)
pca3 = PCA(n_components=min(len(NUM3), 2), random_state=42)
X3_pca = pca3.fit_transform(X3_sc)
cluster_profile3 = df3_p.groupby('Cluster')[NUM3].mean().round(2)

print(" All data & models ready for dashboard.")

# ============================================================
# 2. DASHBOARD 1 – REGRESSION & CORRELATION (DS1)
# ============================================================
print("\n Building Dashboard 1 – Regression & Correlation...")

fig_d1 = make_subplots(
    rows=2, cols=3,
    subplot_titles=(
        'Pearson Correlation with House Value',
        'Model R² Comparison',
        '5-Fold CV R² Scores',
        'Actual vs Predicted (OLS)',
        'Residuals Distribution',
        'Predicted vs Features (Income)'
    ),
    vertical_spacing=0.14,
    horizontal_spacing=0.10
)

# 1-a  Correlation bar
corr_s = pd.Series(corr1).sort_values()
fig_d1.add_trace(go.Bar(
    x=corr_s.values, y=corr_s.index,
    orientation='h',
    marker_color=['tomato' if v < 0 else 'steelblue' for v in corr_s],
    text=[f'{v:.3f}' for v in corr_s],
    textposition='auto',
    showlegend=False
), row=1, col=1)

# 1-b  Model R² comparison
fig_d1.add_trace(go.Bar(
    x=list(r2_models1.keys()),
    y=list(r2_models1.values()),
    marker_color=['steelblue','darkorange','green'],
    text=[f'{v:.4f}' for v in r2_models1.values()],
    textposition='auto',
    showlegend=False
), row=1, col=2)
fig_d1.add_hline(y=0.85, line_dash='dash', line_color='red',
                 annotation_text='Target R²=0.85', row=1, col=2)

# 1-c  CV scores
fig_d1.add_trace(go.Bar(
    x=[f'Fold {i+1}' for i in range(len(cv1))],
    y=cv1,
    marker_color='teal',
    text=[f'{v:.4f}' for v in cv1],
    textposition='auto',
    showlegend=False
), row=1, col=3)
fig_d1.add_hline(y=cv1.mean(), line_dash='dash', line_color='red',
                 annotation_text=f'Mean={cv1.mean():.4f}', row=1, col=3)

# 1-d  Actual vs Predicted
sample_n = min(500, len(y1_te))
idx_s    = np.random.choice(len(y1_te), sample_n, replace=False)
fig_d1.add_trace(go.Scatter(
    x=y1_te.values[idx_s], y=y1_pred_mlr[idx_s],
    mode='markers',
    marker=dict(color='steelblue', size=4, opacity=0.5),
    name='OLS Pred', showlegend=False
), row=2, col=1)
lim_v = [float(y1_te.min()), float(y1_te.max())]
fig_d1.add_trace(go.Scatter(
    x=lim_v, y=lim_v,
    mode='lines', line=dict(color='red', dash='dash'),
    showlegend=False
), row=2, col=1)

# 1-e  Residual histogram
resid1 = y1_te.values - y1_pred_mlr
fig_d1.add_trace(go.Histogram(
    x=resid1, nbinsx=50,
    marker_color='steelblue', opacity=0.75,
    showlegend=False
), row=2, col=2)

# 1-f  Income vs Predicted
inc_vals = df1_c['median_income'].values[
    np.where(np.isin(np.arange(len(df1_c)),
                     np.arange(len(df1_c))[-len(y1_te):]))[0][:sample_n]]
fig_d1.add_trace(go.Scatter(
    x=df1_c['median_income'].sample(sample_n, random_state=42).values,
    y=mlr1.predict(
        sc1.transform(df1_c[FEAT1].sample(sample_n, random_state=42))),
    mode='markers',
    marker=dict(color='darkorange', size=4, opacity=0.5),
    showlegend=False
), row=2, col=3)

fig_d1.update_layout(
    title=dict(
        text='<b>DASHBOARD 1 – Regression & Correlation Analysis</b><br>'
             '<sup>Dataset 1: California Housing Prices</sup>',
        font=dict(size=16), x=0.5
    ),
    height=750, width=1300,
    plot_bgcolor='white',
    paper_bgcolor='#F8F9FA',
    font=dict(family='Arial', size=11)
)
fig_d1.update_xaxes(showgrid=True, gridcolor='#E8E8E8')
fig_d1.update_yaxes(showgrid=True, gridcolor='#E8E8E8')
fig_d1.write_html('dashboard1_regression.html')
fig_d1.show()
print(" Dashboard 1 saved → dashboard1_regression.html")

# ============================================================
# 3. DASHBOARD 2 – PREDICTIVE ANALYTICS (DS2)
# ============================================================
print("\n Building Dashboard 2 – Predictive Analytics...")

rows_d2 = 3 if TS_OK else 2
specs_d2 = [[{}, {}], [{}, {}]]
if TS_OK:
    specs_d2 = [[{'colspan': 2}, None], [{}, {}], [{}, {}]]
    subtitles_d2 = (
        'Monthly Sales Time Series',
        '',
        'RF vs GB – Metrics Comparison',
        'Feature Importance (Random Forest)',
        'Actual vs Predicted (RF)',
        'Cross-Validation R² Scores'
    )
else:
    subtitles_d2 = (
        'RF vs GB – Metrics Comparison',
        'Feature Importance (Random Forest)',
        'Actual vs Predicted (RF)',
        'Cross-Validation R² Scores'
    )

fig_d2 = make_subplots(
    rows=rows_d2, cols=2,
    specs=specs_d2,
    subplot_titles=subtitles_d2,
    vertical_spacing=0.12,
    horizontal_spacing=0.10
)

r_offset = 1

# 3-a  Time Series (if available)
if TS_OK:
    fig_d2.add_trace(go.Scatter(
        x=ts2.index, y=ts2.values,
        mode='lines+markers',
        line=dict(color='steelblue', width=2),
        marker=dict(size=5),
        fill='tozeroy', fillcolor='rgba(70,130,180,0.15)',
        name='Monthly Sales', showlegend=False
    ), row=1, col=1)
    r_offset = 2

# 3-b  RF vs GB metrics bar
metrics_names = ['R²', 'RMSE', 'MAPE(%)']
rf_vals  = [r2_rf2, rmse_rf2, mape_rf2]
gb_vals  = [r2_gb2, rmse_gb2,
            np.mean(np.abs((y2_te.values - y2_gb) /
                           (y2_te.values + 1e-9))) * 100]
fig_d2.add_trace(go.Bar(
    name='Random Forest', x=metrics_names, y=rf_vals,
    marker_color='steelblue',
    text=[f'{v:.3f}' for v in rf_vals], textposition='auto'
), row=r_offset, col=1)
fig_d2.add_trace(go.Bar(
    name='Gradient Boosting', x=metrics_names, y=gb_vals,
    marker_color='darkorange',
    text=[f'{v:.3f}' for v in gb_vals], textposition='auto'
), row=r_offset, col=2)

# 3-c  Feature importance
fig_d2.add_trace(go.Bar(
    x=fi2.values[::-1], y=fi2.index[::-1],
    orientation='h',
    marker_color='teal',
    text=[f'{v:.4f}' for v in fi2.values[::-1]],
    textposition='auto',
    showlegend=False
), row=r_offset, col=2)

# 3-d  Actual vs Predicted RF
sn2 = min(400, len(y2_te))
idx2 = np.random.choice(len(y2_te), sn2, replace=False)
fig_d2.add_trace(go.Scatter(
    x=y2_te.values[idx2], y=y2_rf[idx2],
    mode='markers',
    marker=dict(color='steelblue', size=4, opacity=0.5),
    showlegend=False
), row=r_offset+1, col=1)
lim2 = [float(y2_te.min()), float(y2_te.max())]
fig_d2.add_trace(go.Scatter(
    x=lim2, y=lim2,
    mode='lines', line=dict(color='red', dash='dash'),
    showlegend=False
), row=r_offset+1, col=1)

# 3-e  CV scores
fig_d2.add_trace(go.Bar(
    x=[f'Fold {i+1}' for i in range(len(cv2))],
    y=cv2,
    marker_color='darkorange',
    text=[f'{v:.4f}' for v in cv2],
    textposition='auto',
    showlegend=False
), row=r_offset+1, col=2)
fig_d2.add_hline(y=cv2.mean(), line_dash='dash', line_color='red',
                 annotation_text=f'Mean={cv2.mean():.4f}',
                 row=r_offset+1, col=2)

fig_d2.update_layout(
    title=dict(
        text='<b>DASHBOARD 2 – Predictive Analytics</b><br>'
             '<sup>Dataset 2: Superstore Sales Forecasting</sup>',
        font=dict(size=16), x=0.5
    ),
    height=850, width=1300,
    barmode='group',
    plot_bgcolor='white',
    paper_bgcolor='#F8F9FA',
    font=dict(family='Arial', size=11),
    legend=dict(orientation='h', yanchor='bottom',
                y=1.02, xanchor='right', x=1)
)
fig_d2.update_xaxes(showgrid=True, gridcolor='#E8E8E8')
fig_d2.update_yaxes(showgrid=True, gridcolor='#E8E8E8')
fig_d2.write_html('dashboard2_predictive.html')
fig_d2.show()
print(" Dashboard 2 saved → dashboard2_predictive.html")

# ============================================================
# 4. DASHBOARD 3 – EDA & CUSTOMER SEGMENTATION (DS3)
# ============================================================
print("\n Building Dashboard 3 – EDA & Segmentation...")

fig_d3 = make_subplots(
    rows=2, cols=3,
    subplot_titles=(
        f'K-Means Clusters (k={best_k3})',
        'Silhouette Score vs K',
        'PCA – Customer Distribution',
        'Cluster Profile (Mean Values)',
        'Distribution by Feature',
        'Feature Correlation Heatmap'
    ),
    vertical_spacing=0.14,
    horizontal_spacing=0.10
)

PALETTE3 = px.colors.qualitative.Set1[:best_k3]

# 4-a  Cluster scatter (Income vs Spending)
if len(NUM3) >= 2:
    for cl in range(best_k3):
        mask = df3_p['Cluster'] == cl
        fig_d3.add_trace(go.Scatter(
            x=df3_p.loc[mask, NUM3[0]],
            y=df3_p.loc[mask, NUM3[1]],
            mode='markers',
            name=f'Cluster {cl}',
            marker=dict(color=PALETTE3[cl], size=8,
                        opacity=0.8,
                        line=dict(width=0.5, color='black')),
        ), row=1, col=1)

# 4-b  Silhouette scores
fig_d3.add_trace(go.Scatter(
    x=list(range(2, 9)), y=sil3,
    mode='lines+markers',
    line=dict(color='green', width=2),
    marker=dict(size=8, symbol='circle'),
    showlegend=False
), row=1, col=2)
fig_d3.add_vline(x=best_k3, line_dash='dash', line_color='red',
                 annotation_text=f'Best k={best_k3}', row=1, col=2)

# 4-c  PCA biplot
if X3_pca.shape[1] >= 2:
    for cl in range(best_k3):
        mask = df3_p['Cluster'] == cl
        fig_d3.add_trace(go.Scatter(
            x=X3_pca[mask.values, 0],
            y=X3_pca[mask.values, 1],
            mode='markers',
            name=f'Cluster {cl} (PCA)',
            marker=dict(color=PALETTE3[cl], size=7,
                        opacity=0.75,
                        line=dict(width=0.3, color='black')),
            showlegend=False
        ), row=1, col=3)

# 4-d  Cluster profile bar chart
for i, col in enumerate(NUM3):
    fig_d3.add_trace(go.Bar(
        name=col,
        x=[f'C{cl}' for cl in cluster_profile3.index],
        y=cluster_profile3[col].values,
        text=cluster_profile3[col].round(1).values,
        textposition='auto',
    ), row=2, col=1)

# 4-e  Box plots per numeric feature
colors_box = ['steelblue','darkorange','green','purple','crimson']
for i, col in enumerate(NUM3):
    fig_d3.add_trace(go.Box(
        y=df3_p[col].dropna(),
        name=col,
        marker_color=colors_box[i % len(colors_box)],
        boxmean='sd',
        showlegend=False
    ), row=2, col=2)

# 4-f  Correlation heatmap (as heatmap trace)
corr3 = df3_p[NUM3].corr()
fig_d3.add_trace(go.Heatmap(
    z=corr3.values,
    x=corr3.columns.tolist(),
    y=corr3.index.tolist(),
    colorscale='RdYlGn',
    zmid=0,
    text=corr3.round(3).values,
    texttemplate='%{text}',
    showscale=True,
    colorbar=dict(len=0.35, y=0.15),
    showlegend=False
), row=2, col=3)

fig_d3.update_layout(
    title=dict(
        text='<b>DASHBOARD 3 – Exploratory Data Analysis</b><br>'
             '<sup>Dataset 3: Mall Customer Segmentation</sup>',
        font=dict(size=16), x=0.5
    ),
    height=820, width=1350,
    barmode='group',
    plot_bgcolor='white',
    paper_bgcolor='#F8F9FA',
    font=dict(family='Arial', size=11),
    legend=dict(orientation='v', x=1.01, y=0.85)
)
fig_d3.update_xaxes(showgrid=True, gridcolor='#E8E8E8')
fig_d3.update_yaxes(showgrid=True, gridcolor='#E8E8E8')
fig_d3.write_html('dashboard3_eda.html')
fig_d3.show()
print(" Dashboard 3 saved → dashboard3_eda.html")

# ============================================================
# 5. MASTER EXECUTIVE DASHBOARD (All 3 Datasets Combined)
# ============================================================
print("\n Building Master Executive Dashboard...")

fig_exec = make_subplots(
    rows=3, cols=3,
    subplot_titles=(
        '① Housing: Correlation with Price',
        '② Housing: Model R² Scores',
        '③ Housing: Actual vs Predicted',
        '④ Sales: RF Feature Importance',
        '⑤ Sales: Monthly Revenue Trend',
        '⑥ Sales: CV Performance',
        '⑦ Customers: K-Means Clusters',
        '⑧ Customers: Cluster Profile',
        '⑨ Customers: PCA Biplot'
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.09
)

# ── Row 1 – DS1 ─────
# ①  Correlation
cs = pd.Series(corr1).sort_values()
fig_exec.add_trace(go.Bar(
    x=cs.values, y=cs.index, orientation='h',
    marker_color=['tomato' if v < 0 else '#2196F3' for v in cs],
    text=[f'{v:.3f}' for v in cs], textposition='auto',
    showlegend=False
), row=1, col=1)

# ②  R² scores
fig_exec.add_trace(go.Bar(
    x=list(r2_models1.keys()),
    y=list(r2_models1.values()),
    marker_color=['#2196F3','#FF9800','#4CAF50'],
    text=[f'{v:.4f}' for v in r2_models1.values()],
    textposition='auto', showlegend=False
), row=1, col=2)
fig_exec.add_hline(y=0.85, line_dash='dash',
                   line_color='red', row=1, col=2)

# ③  Actual vs Predicted
fig_exec.add_trace(go.Scatter(
    x=y1_te.values[idx_s], y=y1_pred_mlr[idx_s],
    mode='markers',
    marker=dict(color='#2196F3', size=3, opacity=0.4),
    showlegend=False
), row=1, col=3)
fig_exec.add_trace(go.Scatter(
    x=lim_v, y=lim_v,
    mode='lines', line=dict(color='red', dash='dash'),
    showlegend=False
), row=1, col=3)

# ── Row 2 – DS2 ──
# ④  Feature importance
fig_exec.add_trace(go.Bar(
    x=fi2.values[::-1], y=fi2.index[::-1],
    orientation='h',
    marker_color='#FF9800',
    text=[f'{v:.4f}' for v in fi2.values[::-1]],
    textposition='auto', showlegend=False
), row=2, col=1)

# ⑤  Time series
if TS_OK:
    fig_exec.add_trace(go.Scatter(
        x=ts2.index, y=ts2.values,
        mode='lines+markers',
        line=dict(color='#FF9800', width=2),
        marker=dict(size=4),
        fill='tozeroy',
        fillcolor='rgba(255,152,0,0.15)',
        showlegend=False
    ), row=2, col=2)
else:
    fig_exec.add_trace(go.Bar(
        x=['RF R²', 'RF RMSE', 'RF MAPE'],
        y=[r2_rf2, rmse_rf2, mape_rf2],
        marker_color='#FF9800', showlegend=False
    ), row=2, col=2)

# ⑥  CV scores
fig_exec.add_trace(go.Scatter(
    x=[f'F{i+1}' for i in range(len(cv2))],
    y=cv2,
    mode='lines+markers',
    line=dict(color='#FF9800', width=2),
    marker=dict(size=7),
    showlegend=False
), row=2, col=3)
fig_exec.add_hline(y=cv2.mean(), line_dash='dash',
                   line_color='red', row=2, col=3)

# ── Row 3 – DS3 ────
# ⑦  Cluster scatter
if len(NUM3) >= 2:
    for cl in range(best_k3):
        mask = df3_p['Cluster'] == cl
        fig_exec.add_trace(go.Scatter(
            x=df3_p.loc[mask, NUM3[0]],
            y=df3_p.loc[mask, NUM3[1]],
            mode='markers',
            name=f'Cluster {cl}',
            marker=dict(color=PALETTE3[cl], size=7,
                        opacity=0.8,
                        line=dict(width=0.3, color='black')),
        ), row=3, col=1)

# ⑧  Cluster profile
for col in NUM3:
    fig_exec.add_trace(go.Bar(
        name=col,
        x=[f'C{cl}' for cl in cluster_profile3.index],
        y=cluster_profile3[col].values,
        showlegend=False
    ), row=3, col=2)

# ⑨  PCA biplot
if X3_pca.shape[1] >= 2:
    for cl in range(best_k3):
        mask = df3_p['Cluster'] == cl
        fig_exec.add_trace(go.Scatter(
            x=X3_pca[mask.values, 0],
            y=X3_pca[mask.values, 1],
            mode='markers',
            name=f'Seg {cl}',
            marker=dict(color=PALETTE3[cl], size=6,
                        opacity=0.75,
                        line=dict(width=0.3, color='black')),
            showlegend=False
        ), row=3, col=3)

fig_exec.update_layout(
    title=dict(
        text=(
            '<b>EXECUTIVE DASHBOARD – Comprehensive Data Analytics</b><br>'
            '<sup>Student: Gountante Douti | PRN: 31250500042 | '
            'FY Statistics & Data Sciences – Vishwakarma University</sup>'
        ),
        font=dict(size=17), x=0.5
    ),
    height=1100, width=1400,
    barmode='group',
    plot_bgcolor='white',
    paper_bgcolor='#F0F4F8',
    font=dict(family='Arial', size=10),
    legend=dict(orientation='h', yanchor='bottom',
                y=-0.05, xanchor='center', x=0.5)
)
fig_exec.update_xaxes(showgrid=True, gridcolor='#E0E0E0')
fig_exec.update_yaxes(showgrid=True, gridcolor='#E0E0E0')
fig_exec.write_html('dashboard_EXECUTIVE.html')
pio.show(fig_d1)
print(" MASTER Executive Dashboard saved → dashboard_EXECUTIVE.html")

# ============================================================
# 6. KPI SUMMARY TABLE (Matplotlib static)
# ============================================================
print("\n Building KPI Summary Table...")

kpi_data = {
    'KPI'    : [
        'DS1 – OLS R²', 'DS1 – Ridge R²', 'DS1 – CV R²',
        'DS1 – RMSE ($)', 'DS1 – Logistic Acc.',
        'DS2 – RF R²', 'DS2 – RF RMSE ($)', 'DS2 – RF MAPE (%)',
        'DS2 – GB R²', 'DS2 – CV R²',
        'DS3 – Best K', 'DS3 – Silhouette', 'DS3 – PCA Var (%)'
    ],
    'Value'  : [
        f'{r2_models1["OLS"]:.4f}',
        f'{r2_models1["Ridge"]:.4f}',
        f'{cv1.mean():.4f} ± {cv1.std():.4f}',
        f'{rmse_models1["OLS"]:,.0f}',
        f'{log1_acc:.2%}',
        f'{r2_rf2:.4f}',
        f'{rmse_rf2:,.2f}',
        f'{mape_rf2:.2f}%',
        f'{r2_gb2:.4f}',
        f'{cv2.mean():.4f} ± {cv2.std():.4f}',
        str(best_k3),
        f'{max(sil3):.4f}',
        f'{float(np.cumsum(pca3.explained_variance_ratio_)[-1])*100:.1f}%'
    ],
    'Status' : [
        'OK' if r2_models1['OLS']   > 0.60 else 'Check',
        'OK' if r2_models1['Ridge'] > 0.60 else 'Check',
        'OK' if cv1.mean()          > 0.60 else 'Check',
        'OK',
        'OK' if log1_acc > 0.80 else 'Check',
        'OK' if r2_rf2  > 0.70 else 'Check',
        'OK',
        'OK' if mape_rf2 < 20 else 'Check',
        'OK' if r2_gb2  > 0.70 else 'Check',
        'OK' if cv2.mean() > 0.70 else 'Check',
        'OK', 'OK', 'OK'
    ],
    'Section': [
        'S1','S1','S1','S1','S1',
        'S2','S2','S2','S2','S2',
        'S3','S3','S3'
    ]
}

kpi_df = pd.DataFrame(kpi_data)

fig_kpi, ax = plt.subplots(figsize=(12, 7))
ax.axis('off')
tbl = ax.table(
    cellText=kpi_df.values,
    colLabels=kpi_df.columns,
    cellLoc='center', loc='center',
    colWidths=[0.40, 0.28, 0.10, 0.12]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.2, 1.8)

# Header styling
for j in range(len(kpi_df.columns)):
    tbl[0, j].set_facecolor('#1565C0')
    tbl[0, j].set_text_props(color='white', fontweight='bold')

# Row colors by section
section_colors = {'S1': '#EBF5FB', 'S2': '#E8F8F5', 'S3': '#FEF9E7'}
for i in range(len(kpi_df)):
    sec = kpi_df.iloc[i]['Section']
    for j in range(len(kpi_df.columns)):
        tbl[i+1, j].set_facecolor(section_colors.get(sec, 'white'))

ax.set_title('KEY PERFORMANCE INDICATORS – All Sections',
             fontsize=14, fontweight='bold', pad=20, y=0.98)
plt.tight_layout()
plt.savefig('sec5_kpi_table.png', bbox_inches='tight', dpi=120)
plt.show()
print(" KPI Table saved → sec5_kpi_table.png")

# ============================================================
# 7. FINAL SUMMARY
# ============================================================
print("\n" + "="*65)
print("  SECTION 5 COMPLETE – All Dashboards Generated")
print("="*65)
print("  Interactive HTML Dashboards:")
print("   dashboard1_regression.html   – DS1 Regression")
print("   dashboard2_predictive.html   – DS2 Predictive")
print("   dashboard3_eda.html          – DS3 EDA")
print("   dashboard_EXECUTIVE.html     – Master Dashboard (All 3)")
print("\n  Static Files:")
print("   sec5_kpi_table.png           – KPI Summary Table")
print("="*65)
print("\n ALL 5 SECTIONS COMPLETE.")
print("   Total: Regression | Predictive | EDA | Report | Dashboard")
print("="*65)

